<p class="lead">Este notebook es la edición en R de la sexta unidad del curso. Las diapositivas de clase son un mapa; este texto es el territorio. Las Unidades 2–4 construyeron clasificadores cuya forma estaba fijada de antemano —una frontera recta (regresión logística), una gaussiana por clase (Bayes ingenuo), cortes paralelos a los ejes (árboles)— y la Unidad 5 encontró estructura sin etiquetas. Esta unidad presenta la familia de modelos que hoy domina el campo: las <em>redes neuronales artificiales</em>, que ensamblan muchas unidades simples en una función lo bastante flexible como para representar casi cualquier relación entre entradas y salidas, y que aprenden esa función mediante descenso de gradiente. La unidad sigue en orden las subunidades oficiales: la inspiración biológica (Sección 2), qué es una red (3), para qué sirve y para qué no (4), dónde se usa en medicina (5), el modelo computacional de la neurona (6), el perceptrón simple (7), el perceptrón multicapa y la retropropagación (8), el entrenamiento en la práctica (9) y un ejemplo resuelto completo con los datos de cáncer de mama (10). Cada algoritmo se escribe primero en papel, luego se implementa en unas pocas líneas de R base y solo entonces se compara con los paquetes <code>nnet</code> y <code>neuralnet</code>. No se usa ningún framework de aprendizaje profundo; todo lo que ocurre dentro de la red queda a la vista.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Describir el cerebro humano en cifras (neuronas, sinapsis, velocidad, paralelismo) y explicar qué propiedades toman de él las redes artificiales y cuáles no.</li>
<li>Identificar los componentes de una red neuronal —entradas, pesos, sesgo, suma, función de activación, salida, capas— en un diagrama o en una fórmula, y contar sus parámetros.</li>
<li>Calcular a mano la salida de una neurona de McCulloch–Pitts para las funciones AND y OR, y demostrar geométricamente por qué ninguna neurona individual calcula XOR.</li>
<li>Enunciar la regla de aprendizaje de Rosenblatt y el teorema de convergencia del perceptrón, seguir el algoritmo a mano durante una época y explicar por qué falla con datos no separables.</li>
<li>Escribir la propagación hacia adelante de un perceptrón multicapa como productos de matrices, derivar la retropropagación con la regla de la cadena para una red 2-2-1 y calcular a mano cada derivada parcial para un ejemplo.</li>
<li>Explicar el papel de la tasa de aprendizaje, el número de épocas, la detención temprana, la inicialización de los pesos, la estandarización de las entradas y la regularización, y reconocer en una curva de pérdida la divergencia, el sobreajuste y los gradientes que se desvanecen.</li>
<li>Entrenar, seleccionar y evaluar un perceptrón multicapa en R con <code>nnet</code> usando la metodología del Suplemento S1, y compararlo de forma justa con los métodos de las Unidades 3–4.</li>
<li>Explicar qué agrega lo "profundo" a un perceptrón multicapa y qué cambian las redes convolucionales, recurrentes y basadas en atención, como preparación para la Unidad 8.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código (chunk) es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que interpreta esa salida. Los ejercicios (cajas verdes) te piden modificar el bloque de código que está sobre ellos y volver a ejecutarlo; son la forma más rápida de aprender. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan esta unidad con otras unidades o con el curso de bioinformática. El notebook usa solo R base con <code>stats</code>, <code>MASS</code>, <code>nnet</code>, <code>neuralnet</code>, <code>pROC</code>, <code>mlbench</code>, <code>e1071</code>, <code>rpart</code>, <code>class</code> y <code>randomForest</code>; en un entorno de ejecución de R recién iniciado en Google Colab, el estudiante ejecuta una vez <code>install.packages(c("nnet","neuralnet","pROC","mlbench","e1071","rpart","randomForest"))</code>, y no se descarga ningún dato. Tres algoritmos se siguen a mano antes de que el código los confirme: la neurona que calcula AND y OR (Sección 6.3), la regla de aprendizaje del perceptrón (Sección 7.3) y la retropropagación (Sección 8.4). La metodología de evaluación —conjuntos reservados, validación cruzada, sensibilidad y especificidad, curvas ROC— se da por conocida a partir del Suplemento S1.</p></div>



In [ ]:
# Configuración: bibliotecas usadas a lo largo del notebook. Las semillas están fijas para que cada número citado sea reproducible.
suppressPackageStartupMessages({
  library(nnet); library(neuralnet); library(mlbench); library(MASS); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"; PURPLE <- "#6a3d9a"
sigmoid <- plogis                       # plogis(z) = 1 / (1 + exp(-z)) de R es la sigmoide logística
cat("R", as.character(getRversion()), "| nnet", as.character(packageVersion("nnet")),
    "| neuralnet", as.character(packageVersion("neuralnet")), "\n")



La salida impresa registra las versiones de R y de los paquetes con que se construyeron estos apuntes. En R la sigmoide logística ya existe como `plogis()` (y su inversa, el logit, como `qlogis()`), así que la renombramos como `sigmoid` y nunca implementamos \( 1/(1+e^{-z}) \) a mano: un primer ejemplo de cómo la estadística viene incorporada en el lenguaje.

## 1. Introducción

Hasta ahora, cada clasificador de este curso ha tenido una forma elegida por su diseñador. La regresión logística (Unidad 2) traza una frontera recta; el Bayes ingenuo gaussiano (Unidad 3) supone que cada clase es una campana; un árbol de decisión (Unidad 4) corta el espacio de características en cajas; una máquina de vectores de soporte con kernel radial curva la frontera, pero solo de la manera que el kernel permite. Cuando la verdadera relación entre las características y la etiqueta tiene otra forma, el <em>sesgo</em> del método —el error que comete incluso con datos infinitos— no puede reducirse recolectando más datos. Las redes neuronales artificiales abordan el problema desde la dirección opuesta: ensamblan un gran número de unidades muy simples, cada una de las cuales calcula una suma ponderada seguida de una función no lineal fija, las conectan en capas y dejan que los datos de entrenamiento decidan la forma de la función. El resultado es un modelo que, con suficientes unidades, puede aproximar esencialmente cualquier relación continua (Sección 4), y cuyos parámetros —los <em>pesos</em> de las conexiones— se aprenden todos con el mismo procedimiento: descenso de gradiente sobre una función de pérdida.

La Unidad 1 ubicó esta familia dentro del enfoque <strong>conexionista</strong> de la inteligencia artificial: el conocimiento se almacena de forma implícita, distribuido en los pesos, y se adquiere mediante entrenamiento en lugar de programarse. El enfoque es antiguo. McCulloch y Pitts propusieron la neurona artificial en 1943,<sup class="c"><a href="#r1">[1]</a></sup> Rosenblatt construyó la primera máquina que aprendía, el perceptrón, en 1958,<sup class="c"><a href="#r2">[2]</a></sup> Minsky y Papert demostraron sus límites en 1969,<sup class="c"><a href="#r3">[3]</a></sup> y Rumelhart, Hinton y Williams popularizaron en 1986 el algoritmo que entrena redes de varias capas, la retropropagación.<sup class="c"><a href="#r4">[4]</a></sup> Sin embargo, durante la mayor parte de ese tiempo las redes neuronales fueron un método entre muchos, a menudo superado por las máquinas de vectores de soporte y los bosques aleatorios de las Unidades 3–4 en los pequeños conjuntos de datos tabulares disponibles. El punto de inflexión fue 2012, cuando una red de ocho capas entrenada con un millón de imágenes etiquetadas redujo el error del benchmark de reconocimiento ImageNet por un margen al que ningún otro método se había acercado.<sup class="c"><a href="#r5">[5]</a></sup> Los ingredientes eran los mismos que en 1986; lo que cambió fue la cantidad de datos y de cómputo, y un puñado de detalles de ingeniería (Sección 9) que hicieron que las redes profundas fueran entrenables en la práctica.<sup class="c"><a href="#r6">[6]</a></sup> Desde entonces, las redes neuronales se han convertido en el método por defecto siempre que la entrada es una imagen, una señal, una secuencia o un texto, es decir, siempre que las características no pueden diseñarse a mano.

Esta unidad se detiene antes del aprendizaje "profundo" propiamente tal, que la Unidad 8 aplica a imágenes médicas. Su tema es el <strong>perceptrón multicapa</strong> (MLP): la red más simple con capas ocultas y el componente a partir del cual se construye toda arquitectura moderna. Entender cómo un MLP calcula su salida, cómo la retropropagación calcula el gradiente y qué puede salir mal durante el entrenamiento es todo lo que se necesita para leer críticamente la literatura de aprendizaje profundo.



In [ ]:
# Figura 1: una neurona individual, un perceptrón de una capa y un perceptrón multicapa, dibujados como grafos.
draw_net <- function(sizes, labels, title, components = FALSE) {
  plot.new(); plot.window(xlim = c(0, 1), ylim = c(0, 1.1)); title(main = title, cex.main = 1)
  xs <- seq(0.12, 0.88, length.out = length(sizes)); pos <- list()
  for (li in seq_along(sizes)) {
    n <- sizes[li]; ys <- 0.5 + 0.12 * seq(-(n - 1) / 2, (n - 1) / 2, length.out = n)
    pos[[li]] <- cbind(xs[li], ys)
    text(xs[li], max(ys) + 0.13, labels[li], cex = 0.8)
  }
  for (li in seq_len(length(sizes) - 1)) for (a in seq_len(sizes[li])) for (b in seq_len(sizes[li + 1]))
    segments(pos[[li]][a, 1], pos[[li]][a, 2], pos[[li + 1]][b, 1], pos[[li + 1]][b, 2], col = GREY, lwd = 0.7)
  for (li in seq_along(sizes)) {
    fill <- if (li == 1) "white" else if (li < length(sizes)) BLUE else ORANGE
    points(pos[[li]], pch = 21, bg = fill, cex = 3.2, lwd = 1.3)
  }
  if (components) text(0.5, 0.06, expression(z == sum(w[i] * x[i]) + b ~ "," ~ a == phi(z)), cex = 0.9)
}
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
draw_net(c(3, 1), c("inputs x1..x3", "output a"), "(a) a single neuron", components = TRUE)
draw_net(c(4, 2), c("input layer", "output layer"), "(b) single-layer perceptron")
draw_net(c(3, 4, 2), c("input", "hidden", "output"), "(c) multilayer perceptron 3-4-2")



<p class="caption"><b>Figura 1.</b> Los tres objetos de esta unidad. (a) Una neurona artificial individual: las entradas \(x_i\) llegan por conexiones que llevan pesos \(w_i\); la unidad calcula la suma ponderada \(z = \sum_i w_i x_i + b\) (el sesgo \(b\) es un peso sobre una entrada constante igual a 1) y aplica una función de activación \(\varphi\) para producir la salida \(a\). (b) Un perceptrón de una sola capa: varias neuronas que comparten las mismas entradas; la capa de entrada no realiza ningún cálculo y no se cuenta como capa de neuronas. (c) Un perceptrón multicapa con una capa oculta (azul) y una capa de salida (naranja); cada unidad de una capa está conectada con cada unidad de la siguiente, de ahí que se diga <em>totalmente conectada</em> o <em>densa</em>. La red (c) tiene \(3 \times 4 + 4 + 4 \times 2 + 2 = 26\) parámetros.</p>

## 2. Características generales del cerebro humano

El nombre "red neuronal" es una declaración de parentesco con el cerebro, y esa declaración debe examinarse antes de usarla. Esta sección resume lo que se sabe sobre el órgano, indica cuáles de sus propiedades imitan las redes artificiales de esta unidad y deja explícito cuáles no.

### 2.1 El cerebro en cifras

El cerebro humano adulto contiene unos 86 mil millones de neuronas —\(86.1 \pm 8.1 \times 10^9\) según los recuentos con fraccionador isotrópico de Azevedo y colaboradores—, junto con un número aproximadamente igual de células no neuronales (gliales).<sup class="c"><a href="#r7">[7]</a></sup> La cifra popular de 100 mil millones es una estimación más antigua que los recuentos directos no confirmaron; y, contrariamente a otra afirmación popular, el cerebro humano no es excepcional por su tamaño, sino que es un cerebro de primate escalado linealmente, notable sobre todo por el número de neuronas que concentra en la corteza cerebral (unos 16 mil millones).<sup class="c"><a href="#r8">[8]</a></sup> Cada neurona cortical recibe del orden de miles de sinapsis; los recuentos estereológicos en la neocorteza humana dan alrededor de \(1.5 \times 10^{14}\) sinapsis solo en esa estructura,<sup class="c"><a href="#r9">[9]</a></sup> de modo que el rango habitual de los libros de texto, \(10^{14}\)–\(10^{15}\) sinapsis para todo el cerebro, tiene el orden de magnitud correcto.

### 2.2 La neurona y el potencial de acción

Una neurona tiene tres partes funcionales. Las <strong>dendritas</strong> son prolongaciones ramificadas que reciben señales de otras neuronas; el <strong>soma</strong> (cuerpo celular) las integra; el <strong>axón</strong> es una única fibra larga que lleva la señal de salida hacia otras células y termina en terminales que forman <strong>sinapsis</strong> con las dendritas de las neuronas siguientes. La comunicación a través de una sinapsis es química: las sinapsis excitatorias elevan el potencial de membrana de la célula receptora y las inhibitorias lo reducen; la <em>fuerza</em> de cada sinapsis —cuánto mueve el potencial una señal entrante— es la magnitud biológica a la que corresponden los pesos de una red artificial.

La señal de salida es el <strong>potencial de acción</strong> (espiga): cuando el potencial de membrana al inicio del axón supera un umbral de alrededor de \(-55\) a \(-50\) mV, se abren canales de sodio dependientes de voltaje, el potencial sube hasta unos \(+30\) mV y vuelve en uno o dos milisegundos, y el pulso se propaga a lo largo del axón sin pérdida de amplitud. Hodgkin y Huxley describieron cuantitativamente el mecanismo iónico en 1952, en el trabajo por el que recibieron el Premio Nobel.<sup class="c"><a href="#r10">[10]</a></sup> El potencial de acción es de <em>todo o nada</em>: su tamaño no depende de cuánto se superó el umbral. La información sobre la intensidad de la entrada se transmite, en cambio, mediante la <em>frecuencia</em> de disparo: una neurona estimulada con más fuerza emite más espigas por segundo, hasta un límite fijado por el período refractario de unos pocos milisegundos durante el cual no puede volver a disparar. Este <strong>código de frecuencia</strong> (rate code) es la lectura biológica de la función de activación de la Sección 6: la salida de una unidad artificial es un número continuo que representa una frecuencia de disparo, no una espiga individual. El bloque de código siguiente simula el modelo más simple que captura el mecanismo, la neurona de <em>integración y disparo con fuga</em> (leaky integrate-and-fire) propuesta por Lapicque en 1907,<sup class="c"><a href="#r11">[11]</a></sup> y grafica su frecuencia de disparo en función de la corriente de entrada.



In [ ]:
# Neurona de integración y disparo con fuga: el potencial de membrana integra la corriente de entrada, se fuga de vuelta al reposo
# y emite una espiga (y luego se reinicia) al alcanzar el umbral. La frecuencia de disparo frente a la entrada es la curva f-I.
tau_m <- 20; R_m <- 10; V_rest <- -70; V_th <- -50; V_reset <- -70; t_ref <- 2   # ms, MOhm, mV
dt <- 0.1; Tdur <- 500
lif <- function(I) {                       # devuelve la traza del potencial y los tiempos de espiga para la corriente de entrada I (nA)
  V <- V_rest; refr <- 0; trace <- numeric(Tdur / dt); spikes <- c()
  for (s in seq_len(Tdur / dt)) {
    if (refr > 0) { refr <- refr - dt; V <- V_reset }
    else V <- V + dt / tau_m * (-(V - V_rest) + R_m * I)
    if (V >= V_th) { spikes <- c(spikes, s * dt); V <- 30; refr <- t_ref }
    trace[s] <- V
  }
  list(trace = trace, spikes = spikes)
}
currents <- seq(0, 50, by = 1)
rates <- sapply(currents, function(I) length(lif(I)$spikes) / Tdur * 1000)    # espigas por segundo (Hz)
for (I in c(1.5, 2.5, 4, 8, 50)) cat(sprintf("input %4.1f nA -> firing rate %5.0f Hz\n", I, length(lif(I)$spikes) / Tdur * 1000))
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
tt <- seq(0, Tdur - dt, by = dt)
plot(tt[1:1500], lif(2.5)$trace[1:1500], type = "l", col = BLUE, lwd = 1, xlab = "time (ms)",
     ylab = "membrane potential (mV)", main = "All-or-none spikes; rate tracks input", ylim = c(-75, 35))
lines(tt[1:1500], lif(4)$trace[1:1500], col = ORANGE, lwd = 1)
abline(h = V_th, col = GREY, lty = 2); legend("topright", c("2.5 nA", "4 nA"), col = c(BLUE, ORANGE), lwd = 1, bty = "n", cex = 0.8)
plot(currents, rates, type = "o", pch = 19, cex = 0.5, col = BLUE, xlab = "input current (nA)",
     ylab = "firing rate (Hz)", main = "f-I curve: threshold, then saturation")



<p class="caption"><b>Figura 2.</b> Izquierda: potencial de membrana de una neurona de integración y disparo con fuga para dos corrientes de entrada. Todas las espigas tienen la misma amplitud; una entrada más intensa solo las hace más frecuentes. Derecha: frecuencia de disparo en función de la corriente de entrada. Por debajo de unos 2 nA gana la fuga y la neurona nunca alcanza el umbral (frecuencia 0); por encima, la frecuencia sube abruptamente y luego se curva hacia el techo \(1/t_{\text{ref}} = 500\) Hz impuesto por el período refractario. La curva tiene exactamente la forma de las funciones de activación de la Sección 6: un umbral, una subida pronunciada y una meseta; una sigmoide biológica.</p>

### 2.3 Lento, paralelo, distribuido, robusto, plástico

Hay cinco propiedades del cerebro como dispositivo de cómputo que los modelos conexionistas se proponen capturar.

**Es lento pero masivamente paralelo.** Una neurona dispara a lo sumo unos cientos de veces por segundo y una señal cruza una sinapsis en alrededor de un milisegundo, unas \(10^6\) veces más lento de lo que conmuta un transistor. Aun así, una persona reconoce un rostro en unos 100–150 ms, es decir, en a lo sumo cien pasos en serie. El cerebro compensa la lentitud de sus componentes teniendo \(10^{11}\) de ellos trabajando a la vez; cualquiera que sea el algoritmo que ejecuta, debe ser poco profundo y ancho, no profundo y serial. Las redes artificiales heredan esto: cada capa se calcula en paralelo como un único producto de matrices, razón por la cual los procesadores gráficos (Sección 11) les resultan adecuados.

**El conocimiento está distribuido.** Ninguna neurona individual almacena el concepto "abuela"; un concepto es un patrón de actividad sobre muchas neuronas, y una neurona participa en muchos patrones. Del mismo modo, en una red artificial ningún peso significa algo por sí solo; el conocimiento de una red entrenada está repartido entre todos sus pesos, lo que está en la raíz de su opacidad (Sección 4).

**Se degrada con suavidad.** Las neuronas mueren a diario durante toda la vida adulta, y el cerebro tolera la pérdida de una fracción considerable antes de que la función se vea perceptiblemente afectada; el daño produce un declive gradual en lugar de un colapso. La Sección 4.3 muestra la misma propiedad en una red artificial a la que se le eliminan pesos al azar.

**Aprende modificando sinapsis.** La fuerza de una sinapsis cambia con el uso. Hebb propuso la regla en 1949: cuando una neurona presináptica participa repetidamente en el disparo de una postsináptica, la conexión entre ellas se fortalece; en la consigna, <em>las células que disparan juntas se conectan juntas</em>.<sup class="c"><a href="#r12">[12]</a></sup> La regla de Hebb es la antecesora de todas las reglas de aprendizaje de esta unidad, y el bloque de código siguiente la implementa en tres líneas.

**Construye sus propias características.** La corteza visual es una jerarquía: las áreas tempranas responden a bordes orientados, y las posteriores a contornos, texturas, partes y, finalmente, objetos y rostros. Nadie diseña estas características; emergen de la exposición al mundo visual. Las redes profundas de la Unidad 8 son su contraparte artificial, y los filtros que aprenden en su primera capa se parecen lo suficiente a los detectores de bordes de la corteza visual primaria como para que el parecido sea una de las ilustraciones clásicas del campo.<sup class="c"><a href="#r6">[6]</a></sup>



In [ ]:
# Regla de Hebb: dw_i = eta * x_i * y. El peso de una entrada que dispara JUNTO CON la salida crece;
# el peso de una entrada que dispara independientemente de la salida apenas crece.
set.seed(0); steps <- 500
post     <- runif(steps) < 0.3                                       # la salida dispara en ~30% de los pasos
pre_corr <- ifelse(post, runif(steps) < 0.9, runif(steps) < 0.05)    # la entrada 1 suele disparar JUNTO CON la salida
pre_rand <- runif(steps) < 0.3                                       # la entrada 2 dispara de forma independiente
eta <- 0.01; w <- c(0, 0)
for (s in seq_len(steps)) w <- w + eta * c(pre_corr[s], pre_rand[s]) * post[s]
tab <- data.frame(input = c("input 1 (correlated)", "input 2 (independent)"),
                  P_fires = round(c(mean(pre_corr), mean(pre_rand)), 3),
                  P_fires_given_output = round(c(mean(pre_corr[post]), mean(pre_rand[post])), 3),
                  final_weight = round(w, 3))
print(tab, row.names = FALSE)



Ambas entradas disparan alrededor del 30 % del tiempo, de modo que una regla que fortaleciera una sinapsis cada vez que la <em>entrada</em> estuviera activa las trataría por igual. La regla de Hebb está condicionada a la coincidencia entre entrada y salida, y la entrada que dispara junto con la salida en cerca del 90 % de las espigas de esta termina con un peso entre dos y tres veces mayor que el de la entrada que coincide con ella solo por azar. La red ha aprendido una <em>correlación</em> a partir de la experiencia, sin un profesor: es el mecanismo que la regla del perceptrón de la Sección 7 convierte en aprendizaje supervisado al agregar una señal de error.

### 2.4 Cerebro y computador, lado a lado

| Propiedad | Cerebro humano | Computador digital |
|---|---|---|
| Elemento básico | neurona (~\(10^{11}\)) con ~\(10^3\)–\(10^4\) sinapsis cada una | transistor (~\(10^{10}\) por chip) |
| Tiempo de conmutación | ~1 ms | ~0.1 ns |
| Funcionamiento | analógico, ruidoso, masivamente paralelo | digital, exacto, mayormente serial |
| Memoria | distribuida en la fuerza de las sinapsis; direccionable por contenido | localizada en direcciones; basada en direcciones |
| Programación | aprendida de la experiencia; plasticidad sináptica | escrita explícitamente por un programador |
| Tolerancia a fallos | degradación suave; tolera la pérdida diaria de neuronas | un solo bit erróneo puede hacer fallar un programa |
| Energía | ~20 W | cientos de watts para una GPU de entrenamiento |

<div class="box warn"><h4>Inspiradas en el cerebro, no un modelo de él</h4><p>Las redes neuronales artificiales toman del cerebro las ideas de unidades simples, conexiones ponderadas, capas, representación distribuida y aprendizaje mediante ajuste de pesos, y nada más. Las unidades de esta unidad no tienen espigas, ni tiempo, ni cómputo dendrítico, ni neurotransmisores, ni glía; su regla de aprendizaje, la retropropagación, requiere que cada sinapsis conozca el error en la salida y se ajuste con la misma señal global, algo para lo cual no se conoce ningún mecanismo biológico; y su entrenamiento necesita millones de ejemplos etiquetados donde a un niño le bastan unos pocos. Una red neuronal es un modelo estadístico ajustado por optimización, y el vocabulario de "neuronas" y "aprendizaje" es una metáfora histórica. Afirmaciones como "la red piensa" o "el algoritmo aprende como un cerebro" deben leerse teniendo esto en cuenta.<sup class="c"><a href="#r6">[6]</a></sup></p></div>

## 3. Qué es una red neuronal artificial

### 3.1 Unidades, pesos, capas, arquitectura

Una <strong>red neuronal artificial</strong> es un grafo dirigido cuyos nodos son <em>unidades</em> (neuronas) y cuyas aristas son <em>conexiones</em>, cada una con un <em>peso</em> de valor real. Todas las unidades calculan lo mismo: la suma ponderada de sus entradas más un <em>sesgo</em> (bias), pasada por una <em>función de activación</em> (Sección 6). Las unidades se organizan en <strong>capas</strong>: una <em>capa de entrada</em>, que simplemente contiene el vector de características y no realiza ningún cálculo; cero o más <em>capas ocultas</em>, llamadas así porque sus valores no se entregan como datos ni se leen como predicciones; y una <em>capa de salida</em>, cuyos valores son la predicción. La <strong>arquitectura</strong> es la especificación de cuántas capas hay, cuántas unidades tiene cada una, qué activación usa cada una y cómo se conectan las capas. La notación 9-16-1 significa 9 entradas, una capa oculta de 16 unidades y una salida; por convención, el número de capas de una red cuenta las capas que calculan, de modo que una red 9-16-1 es una red de <em>dos capas</em> con <em>una capa oculta</em>. Los libros de texto difieren en este conteo —algunos incluyen la capa de entrada—, así que, cuando importe, indica la arquitectura de forma explícita.

Existen dos tipos de arquitectura. En una red <strong>prealimentada</strong> (feed-forward) el grafo no tiene ciclos: la información fluye desde la capa de entrada, a través de las capas ocultas, hasta la salida, y la salida es función solo de la entrada actual. Todas las redes de esta unidad son prealimentadas. En una red <strong>recurrente</strong> algunas conexiones vuelven hacia atrás, de modo que la salida en un paso de tiempo depende de las entradas en pasos anteriores; las redes recurrentes procesan secuencias —habla, electrocardiogramas, texto— y la Sección 11 vuelve sobre ellas.

### 3.2 La red como función parametrizada

Matemáticamente, una red prealimentada con parámetros \(\boldsymbol{\theta}\) (todos sus pesos y sesgos) es una función \(f(\mathbf{x}; \boldsymbol{\theta})\) que va del vector de características \(\mathbf{x} \in \mathbb{R}^p\) a una salida \(\hat{\mathbf{y}}\). La arquitectura fija la <em>forma</em> de \(f\); el entrenamiento elige los <em>valores</em> de \(\boldsymbol{\theta}\) que hacen que \(f(\mathbf{x}_i; \boldsymbol{\theta}) \approx y_i\) en la muestra de entrenamiento, minimizando una función de pérdida con descenso de gradiente: exactamente la receta de la Unidad 2 para la regresión lineal y logística, con una \(f\) más flexible. Una red es un modelo, en el vocabulario de la Unidad 1, y la retropropagación es el algoritmo que lo ajusta.

El número de parámetros es lo primero que hay que calcular al leer una arquitectura, porque acota la cantidad de datos que la red necesita. Para una red totalmente conectada con tamaños de capa \(n_0, n_1, \dots, n_L\) (con \(n_0 = p\)), cada capa \(l\) tiene una matriz de pesos de \(n_{l-1} \times n_l\) y \(n_l\) sesgos, de modo que

<div class="eq">\[ \#\boldsymbol{\theta} = \sum_{l=1}^{L} \left( n_{l-1}\, n_l + n_l \right). \]</div>



In [ ]:
# Conteo de parámetros a partir de la arquitectura: pesos más sesgos de cada capa que calcula.
n_parameters <- function(sizes) sum(head(sizes, -1) * tail(sizes, -1) + tail(sizes, -1))
archs <- list("single neuron, 2 inputs" = c(2, 1),
              "single neuron, 9 inputs (breast cancer)" = c(9, 1),
              "2-2-1 (the XOR network, Section 8)" = c(2, 2, 1),
              "9-16-1 (breast cancer, Section 8)" = c(9, 16, 1),
              "64-16-5 (synthetic glyphs, Section 9)" = c(64, 16, 5),
              "784-500-500-10 (handwritten digits at 28x28)" = c(784, 500, 500, 10))
data.frame(layer_sizes = sapply(archs, paste, collapse = "-"),
           parameters  = sapply(archs, n_parameters), row.names = names(archs))



Una neurona individual con 9 entradas tiene 10 parámetros, el mismo número que una regresión logística con 9 características, y no es una coincidencia (siguiente subsección). Agregar una capa oculta de 16 unidades multiplica el conteo por alrededor de 16; una modesta red de tres capas para imágenes de 28 × 28 ya tiene unos 650,000 parámetros, más que el número de imágenes de entrenamiento de la mayoría de los conjuntos de datos médicos. La razón entre parámetros y casos de entrenamiento es la magnitud que hay que tener presente a lo largo de las Secciones 8–10: una red con más parámetros que casos puede memorizar perfectamente la muestra de entrenamiento, y solo la metodología del Suplemento S1 revela si ha aprendido algo más.

### 3.3 Una neurona individual con sigmoide es una regresión logística

La red más simple —una unidad, activación sigmoide— calcula \(\hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b)\) con \(\sigma(z) = 1/(1+e^{-z})\). Ese es exactamente el modelo de regresión logística de la Unidad 2: los pesos son los coeficientes de la regresión, el sesgo es el intercepto y, si la unidad se entrena con descenso de gradiente sobre la pérdida de entropía cruzada, debe llegar a la misma solución, porque la pérdida es convexa y tiene un único mínimo. El bloque de código siguiente entrena la neurona a mano con dos características estandarizadas de los datos de cáncer de mama<sup class="c"><a href="#r13">[13]</a></sup> y compara sus pesos con los de `glm` de R ajustado como regresión logística.



In [ ]:
# Cáncer de mama de Wisconsin desde mlbench (el sustituto en R de load_breast_cancer de sklearn).
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                   # elimina la columna Id
bc <- bc[complete.cases(bc), ]                             # elimina las 16 filas con valores faltantes
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # 9 factores ordinales -> numéricos
y_mal <- as.numeric(bc$Class == "malignant")               # 1 = maligno, 0 = benigno
feat2 <- c("Cell.size", "Bare.nuclei"); Z2 <- scale(as.matrix(bc[, feat2]))

w <- c(0, 0); b <- 0                                        # una neurona sigmoide con descenso de gradiente de lote completo
for (it in 1:30000) { p <- sigmoid(Z2 %*% w + b); w <- w - 1.0 * t(Z2) %*% (p - y_mal) / length(y_mal); b <- b - mean(p - y_mal) }
g <- glm(y_mal ~ Z2, family = binomial)                    # el mismo modelo ajustado con glm de R (IRLS)
cat(sprintf("sigmoid neuron : w = (%.3f, %.3f)  b = %.3f\n", w[1], w[2], b))
cat(sprintf("glm (logistic) : w = (%.3f, %.3f)  b = %.3f\n", coef(g)[2], coef(g)[3], coef(g)[1]))
cat(sprintf("identical predictions on the %d cases: %.3f\n", nrow(Z2),
            mean((sigmoid(Z2 %*% w + b) >= 0.5) == (predict(g, type = "response") >= 0.5))))



Los pesos coinciden hasta el tercer decimal y los dos modelos hacen la misma predicción para todos los casos: la neurona programada a mano y `glm` ajustan el mismo modelo convexo y llegan al mismo mínimo por caminos numéricos distintos (descenso de gradiente frente a mínimos cuadrados reponderados iterativamente). Por lo tanto, todo lo que sabes sobre la regresión logística —cómo leer un coeficiente, estandarizar las características, la frontera recta— se aplica a una neurona individual. Más importante para esta unidad: una red con capas ocultas es una regresión logística <em>sobre características que la propia red ha aprendido</em>. La unidad de salida de la red 9-16-1 de la Sección 8 es una regresión logística con 16 entradas, pero esas 16 entradas no son características medidas; son combinaciones no lineales de las 9 mediciones, elegidas por el entrenamiento para que la regresión logística final sea exacta. Esto es lo que significa "aprender características", y es la idea más importante de la unidad.

<div class="box key"><h4>Idea clave</h4><p>Una red neuronal es una función parametrizada \(f(\mathbf{x}; \boldsymbol{\theta})\) cuya capa de salida es un modelo conocido (regresión logística para clasificación, regresión lineal para regresión) y cuyas capas ocultas calculan las características que ese modelo recibe. El entrenamiento ajusta todos los parámetros a la vez mediante descenso de gradiente, de modo que las características se aprenden para la tarea en lugar de ser diseñadas por el analista.</p></div>

## 4. Ventajas y limitaciones

### 4.1 Qué hacen bien las redes

**No linealidad y aproximación universal.** Con una función de activación no lineal en la capa oculta, un MLP puede representar funciones que ninguna recta, gaussiana o caja puede representar. El resultado tiene una forma precisa: Cybenko (1989) demostró que una red con una capa oculta de unidades sigmoides y una salida lineal puede aproximar cualquier función continua sobre un dominio acotado con la exactitud que se desee, siempre que tenga suficientes unidades ocultas;<sup class="c"><a href="#r14">[14]</a></sup> Hornik, Stinchcombe y White extendieron el resultado a cualquier activación no constante, acotada y continua.<sup class="c"><a href="#r15">[15]</a></sup> Dos precauciones son esenciales. El teorema garantiza la <em>existencia</em>, no la <em>aprendibilidad</em>: no dice cuántas unidades se necesitan (el número puede ser astronómicamente grande), ni que el descenso de gradiente vaya a encontrar los pesos correctos, ni que una red que ajusta los datos de entrenamiento vaya a generalizar. Y "basta con una capa oculta" es una afirmación sobre la representación, no sobre la eficiencia; la Sección 11 explica por qué la profundidad ayuda en la práctica. El bloque de código siguiente muestra el teorema en acción sobre una señal sintética, usando `nnet` con una salida lineal.

**Características aprendidas.** La Sección 3.3 lo dejó claro: las capas ocultas son extractores de características entrenados para la tarea. Para datos tabulares con un puñado de características diseñadas esto importa poco; para imágenes, señales y secuencias, donde nadie sabe cómo diseñar las características, es decisivo.<sup class="c"><a href="#r6">[6]</a></sup>

**Paralelismo y degradación suave.** Cada unidad de una capa calcula de forma independiente, de modo que una capa es un único producto de matrices, y una red con millones de parámetros se ejecuta en milisegundos en hardware paralelo. Y como el conocimiento está distribuido entre muchos pesos, ningún peso individual es crítico: eliminar una fracción de ellos degrada el desempeño de forma gradual y no abrupta (Sección 4.3), una propiedad que no comparte ningún árbol de decisión, en el que cortar un nodo elimina un subárbol completo.



In [ ]:
# Aproximación universal: una red tanh/sigmoide de una capa oculta (nnet) ajusta un latido de ECG sintético
# (onda P, complejo QRS y onda T como protuberancias gaussianas). Más unidades ocultas -> mejor ajuste, hasta cierto punto.
ecg_beat <- function(t) {
  g <- function(mu, s, a) a * exp(-((t - mu) / s)^2)
  g(0.20, 0.045, 0.15) + g(0.49, 0.015, -0.10) + g(0.52, 0.020, 1.0) + g(0.555, 0.015, -0.25) + g(0.78, 0.06, 0.30)
}
t <- seq(0, 1, length.out = 300); signal <- ecg_beat(t); Xt <- matrix((t - 0.5) * 6, ncol = 1)
par(mfrow = c(1, 4), mar = c(4, 3.2, 2.5, 1))
for (h in c(2, 4, 16, 64)) {
  set.seed(0)
  net <- nnet(Xt, signal, size = h, linout = TRUE, maxit = 2000, trace = FALSE, MaxNWts = 5000, abstol = 1e-8)
  pred <- predict(net, Xt); rmse <- sqrt(mean((pred - signal)^2))
  cat(sprintf("%3d sigmoid hidden units: %4d parameters, RMSE = %.4f\n", h, n_parameters(c(1, h, 1)), rmse))
  plot(t, signal, type = "l", col = GREY, lwd = 3, xlab = "time (s)", ylab = if (h == 2) "amplitude (mV)" else "",
       main = sprintf("%d hidden units\nRMSE %.3f", h, rmse)); lines(t, pred, col = BLUE, lwd = 1.4)
}



<p class="caption"><b>Figura 3.</b> Una red de una capa oculta (<code>nnet</code>, unidades ocultas logísticas, salida lineal) ajustada a un latido de ECG sintético con 2, 4, 16 y 64 unidades. Dos unidades solo pueden colocar dos escalones suaves y omiten por completo el complejo QRS; cuatro capturan toscamente la onda R; dieciséis reproducen el latido completo con un error pequeño, y sesenta y cuatro con uno aún menor. Sin embargo, no está garantizado que el ajuste mejore con el ancho: el optimizador BFGS de <code>nnet</code> puede detenerse en un mínimo local de una pérdida no convexa, de modo que una red más ancha a veces no ajusta mejor que una más angosta, lo que ilustra la segunda limitación de la Sección 4.2: el teorema promete que los pesos existen, no que el entrenamiento los encontrará.</p>

### 4.2 Qué hacen mal las redes

**Necesitan muchos datos.** La flexibilidad que permite a una red representar cualquier función también le permite ajustar el ruido. Una red con \(10^5\) parámetros entrenada con \(10^2\) pacientes los memorizará; los métodos de la Sección 9 (detención temprana, regularización) mitigan el problema, pero no lo eliminan. En conjuntos de datos tabulares pequeños —la cohorte clínica típica de unos pocos cientos de casos—, los bosques aleatorios y las máquinas de vectores de soporte (Unidades 3–4) con frecuencia igualan o superan a las redes, y la Sección 10 pone esto a prueba honestamente con los datos de cáncer de mama.

**Son opacas.** Un árbol entrenado es una regla legible; una red entrenada es un conjunto de unos pocos miles de números cuyo significado individual es nulo. Los métodos post hoc —importancia por permutación (Sección 10), mapas de saliencia (Unidad 8)— ayudan, pero ninguno entrega una explicación que un clínico pueda verificar con la fisiología, y los marcos regulatorios de la Unidad 1 la exigen cada vez más.

**Tienen muchos hiperparámetros.** La arquitectura, la función de activación, la tasa de aprendizaje o el optimizador, el decaimiento de pesos (weight decay), el número de épocas, la inicialización: cada una es una decisión sin teoría que la guíe, y cada una interactúa con las demás. La Sección 9 las revisa; en la práctica se eligen por validación cruzada (Suplemento S1), con un costo en tiempo de cómputo en el que los métodos más simples no incurren.

**El entrenamiento es no convexo.** La pérdida de la regresión logística tiene un único mínimo; la pérdida de una red con capas ocultas tiene muchos mínimos locales, puntos de silla y mesetas planas, de modo que el resultado del entrenamiento depende de los pesos iniciales y del orden de los datos (de ahí `set.seed`), y dos ejecuciones pueden dar modelos distintos y de distinta calidad. La Figura 3 mostró un pequeño ejemplo.

**Son frágiles fuera de la distribución de entrenamiento.** Una red extrapola con confianza hacia regiones donde no ha visto datos; las fallas por aprendizaje de atajos (shortcut learning) y por cambio en la distribución de los datos (dataset shift) de la Unidad 1 son, en la práctica, fallas de redes desplegadas más allá de sus datos de entrenamiento.

### 4.3 La degradación suave, medida

El bloque de código siguiente entrena redes con un conjunto de glifos simulados de 8 × 8 (un sustituto de las imágenes médicas; R no tiene un conjunto de datos `digits`, así que generamos cinco plantillas con ruido), luego elimina al azar una fracción creciente de los pesos de su primera capa —el equivalente artificial de perder neuronas— y mide la exactitud que se conserva. Se comparan dos anchos. El vector de pesos de `nnet` comienza con los pesos de la primera capa, así que poner a cero una fracción aleatoria de sus primeras entradas y volver a evaluar con `maxit = 0` realiza la eliminación.



In [ ]:
# Glifos simulados de 8x8 para cinco clases: una plantilla por clase más ruido gaussiano (el sustituto en R de load_digits).
set.seed(0)
templates <- list(
  vbar = { m <- matrix(0, 8, 8); m[, 4:5] <- 1; m },
  hbar = { m <- matrix(0, 8, 8); m[4:5, ] <- 1; m },
  diag = { m <- matrix(0, 8, 8); for (i in 1:8) m[i, i] <- 1; for (i in 1:7) m[i, i + 1] <- 1; m },
  box  = { m <- matrix(0, 8, 8); m[2:7, 2] <- 1; m[2:7, 7] <- 1; m[2, 2:7] <- 1; m[7, 2:7] <- 1; m },
  blob = { m <- matrix(0, 8, 8); m[3:6, 3:6] <- 1; m })
make_digits <- function(n_per = 150, noise = 0.9) {
  X <- do.call(rbind, lapply(templates, function(tp)
    t(replicate(n_per, pmax(0, pmin(1, as.vector(tp) + rnorm(64, 0, noise)))))))
  list(X = X, y = factor(rep(0:4, each = n_per)))
}
dig <- make_digits(); cat("simulated glyphs:", nrow(dig$X), "images x", ncol(dig$X), "pixels,",
                          nlevels(dig$y), "classes\n")
set.seed(1); tr_d <- sample(nrow(dig$X), round(0.7 * nrow(dig$X)))
Zd <- scale(dig$X); Zd_tr <- Zd[tr_d, ]; Zd_te <- Zd[-tr_d, ]; yd_tr <- dig$y[tr_d]; yd_te <- dig$y[-tr_d]
Ytr <- class.ind(yd_tr)

# Elimina una fracción aleatoria de los pesos entrenados de la primera capa y mide la exactitud en datos reservados.
fractions <- seq(0, 0.9, by = 0.05); set.seed(0); results <- list()
for (h in c(8, 64)) {
  set.seed(0); net <- nnet(Zd_tr, Ytr, size = h, softmax = TRUE, maxit = 300, trace = FALSE, MaxNWts = 8000)
  n_first <- h * (ncol(Zd_tr) + 1)                        # los pesos de la primera capa son las primeras entradas de net$wts
  accs <- sapply(fractions, function(fr) {
    w <- net$wts; kill <- sample(n_first, round(fr * n_first)); w[kill] <- 0
    net0 <- nnet(Zd_tr, Ytr, size = h, softmax = TRUE, Wts = w, maxit = 0, trace = FALSE, MaxNWts = 8000)
    mean((max.col(predict(net0, Zd_te)) - 1) == (as.integer(yd_te) - 1))
  })
  results[[as.character(h)]] <- accs
  cat(sprintf("%2d hidden units: intact %.3f | 30%% deleted %.3f | 60%% deleted %.3f\n", h, accs[1], accs[7], accs[13]))
}
plot(fractions * 100, results[["64"]], type = "o", pch = 19, cex = 0.6, col = BLUE, ylim = c(0, 1),
     xlab = "first-layer weights deleted (%)", ylab = "test accuracy", main = "Graceful degradation of a trained network")
lines(fractions * 100, results[["8"]], type = "o", pch = 19, cex = 0.6, col = ORANGE)
abline(h = 0.2, col = GREY, lty = 3); text(5, 0.24, "chance (5 classes)", col = GREY, cex = 0.8)
legend("bottomleft", c("64 hidden units", "8 hidden units"), col = c(BLUE, ORANGE), lwd = 1, pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 4.</b> Exactitud en prueba de dos redes entrenadas a medida que una fracción aleatoria de los pesos de su primera capa se lleva a cero. El declive es gradual: la red más ancha conserva la mayor parte de su exactitud aun con una gran fracción de sus conexiones eliminadas, e incluso la angosta tolera una pérdida considerable antes de que la exactitud caiga hacia el nivel del azar. La redundancia de la representación distribuida absorbe el daño. Compárese con un programa, en el que una sola instrucción corrupta detiene la ejecución, o con un árbol de decisión, en el que eliminar el nodo raíz elimina todas las decisiones.</p>

<div class="box key"><h4>Ejercicio 1</h4><p>En el bloque de código anterior, elimina pesos de la capa de <em>salida</em> (las últimas entradas de <code>net$wts</code>) en lugar de la primera. La capa de salida tiene muchos menos pesos; ¿es la red igual de tolerante a perderlos? Luego repite el experimento con un árbol <code>rpart</code> de la Unidad 4, podando nodos internos al azar, y compara la forma de la curva.</p></div>

## 5. Aplicaciones en medicina

Las redes neuronales entraron en la medicina mucho antes que el aprendizaje profundo. Baker y colaboradores entrenaron en 1995 una red de tres capas para predecir la malignidad del cáncer de mama a partir de los descriptores mamográficos estandarizados del léxico BI-RADS, y reportaron un área bajo la curva ROC mayor que la de los radiólogos tratantes en los mismos casos: un ejemplo temprano del patrón que el curso ha subrayado, una red ajustada a características diseñadas por clínicos y evaluada con la metodología ROC del Suplemento S1.<sup class="c"><a href="#r16">[16]</a></sup> Hacia 2006 la literatura era lo bastante amplia como para que Lisboa y Taktak la revisaran sistemáticamente en el ámbito del apoyo a la decisión en cáncer: de 396 estudios, solo 27 eran ensayos clínicos o ensayos controlados aleatorizados, y de ellos 21 mostraron un aumento del beneficio para la atención de salud, 6 no mostraron ninguno y ninguno mostró una disminución; un veredicto cautelosamente positivo y un recordatorio de que la mayoría de los modelos publicados nunca se evaluaron de un modo que pudiera demostrar beneficio.<sup class="c"><a href="#r17">[17]</a></sup>

La era del aprendizaje profundo produjo los sistemas que la Unidad 1 citó como evidencia de un desempeño de nivel especialista, todos ellos redes multicapa del tipo que describe esta unidad, escaladas y dotadas de capas convolucionales para imágenes (Unidad 8):

| Año | Tarea | Entrada | Arquitectura | Desempeño reportado | Ref. |
|---|---|---|---|---|---|
| 2016 | Retinopatía diabética | fotografías de fondo de ojo | red convolucional profunda | sensibilidad 97.5 %, especificidad 93.4 % en el punto de operación de alta sensibilidad (EyePACS-1) | <sup class="c"><a href="#r18">[18]</a></sup> |
| 2017 | Clasificación de cáncer de piel | fotografías dermatoscópicas y clínicas | red convolucional profunda | a la par de 21 dermatólogos en imágenes confirmadas por biopsia | <sup class="c"><a href="#r19">[19]</a></sup> |
| 2018 | Decisiones de derivación a partir de OCT retiniana | tomografía de coherencia óptica 3-D | dos redes: segmentación y luego clasificación | tasa de error de 5.5 %, igual a la de especialistas en retina en 997 exámenes | <sup class="c"><a href="#r20">[20]</a></sup> |
| 2019 | Detección de arritmias | ECG ambulatorio de una sola derivación | red convolucional de 34 capas | F1 0.837, superior al cardiólogo promedio en 12 clases de ritmo | <sup class="c"><a href="#r21">[21]</a></sup> |

Tres lecciones corresponden a esta unidad. Primero, cada uno de estos sistemas es un clasificador del tipo que se construye en la Sección 10 —entradas, capas ocultas, salida softmax, pérdida de entropía cruzada, descenso de gradiente— y se diferencia de la red 9-16-1 en la escala y en el uso de la convolución. Segundo, sus entradas son píxeles o voltajes en bruto: la ventaja de las <em>características aprendidas</em> de la Sección 4.1 es lo que los hizo posibles, y es la razón por la que los datos de cáncer de mama, con nueve características diseñadas a mano, no favorecen a las redes (Sección 10). Tercero, como subrayaron Rajkomar, Dean y Kohane en su revisión para clínicos, el desempeño en un conjunto de prueba curado es el comienzo de la evidencia, no el final: la validación prospectiva, la calibración, el cambio en la distribución de los datos (dataset shift) y la integración en el flujo de trabajo deciden si un modelo ayuda a los pacientes.<sup class="c"><a href="#r22">[22]</a></sup>

<div class="box bridge"><h4>Puente con la bioinformática</h4><p>El avance que mejor conocen los estudiantes del curso de bioinformática, AlphaFold, es una red neuronal: sus entradas son alineamientos múltiples de secuencias y características de pares de residuos, y su salida es una estructura proteica. Los modelos QSAR de actividad molecular —los descriptores moleculares como capa de entrada y la actividad como salida— estuvieron entre las primeras aplicaciones de los perceptrones multicapa en las ciencias de la vida, y la metodología de validación cruzada que se usa allí es la que aplica la Sección 10.</p></div>

## 6. El modelo computacional de la neurona

### 6.1 De McCulloch–Pitts a la unidad moderna

McCulloch y Pitts abstrajeron la neurona de la Sección 2.2 en un dispositivo lógico: entradas binarias, cada una excitatoria o inhibitoria, un umbral y una salida binaria que se activa cuando el número de entradas excitatorias activas alcanza el umbral y ninguna entrada inhibitoria está activa. Demostraron que redes de tales unidades pueden calcular cualquier función booleana y, con ello, que la lógica del cerebro podía, en principio, ser realizada por neuronas.<sup class="c"><a href="#r1">[1]</a></sup> Su unidad no aprendía. El perceptrón de Rosenblatt (Sección 7) dio a cada entrada un <em>peso</em> de valor real ajustable a partir de los datos, y la unidad moderna generaliza el umbral a una <em>función de activación</em> arbitraria. El cálculo de una unidad con \(p\) entradas se hace en dos pasos:

<div class="eq">\[ z = \sum_{i=1}^{p} w_i x_i + b = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \varphi(z), \]
<span class="where">donde \(z\) es la <em>preactivación</em> (entrada neta), \(w_i\) el peso de la entrada \(i\), \(b\) el sesgo, \(\varphi\) la función de activación y \(a\) la <em>activación</em> (salida) de la unidad. El sesgo puede incorporarse a los pesos agregando una entrada constante \(x_0 = 1\) con peso \(w_0 = b\), que es como suele dibujarse y como lo trata el código de la Sección 7.</span></div>

Geométricamente, \(z = 0\) es un hiperplano en el espacio de características —una recta cuando \(p = 2\)— y el signo de \(z\) indica en qué lado de él se encuentra \(\mathbf{x}\). Por lo tanto, una unidad individual divide el espacio de características en dos semiespacios, cualquiera que sea la activación que use; la activación solo determina cómo varía la salida con la distancia a la frontera.

### 6.2 Funciones de activación

La función de activación es lo que hace que una red sea más que un modelo lineal: sin ella, una pila de capas calcula \(\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2\), que vuelve a ser una función lineal de \(\mathbf{x}\), y ningún número de capas agregaría nada. Cinco funciones cubren casi toda la práctica.

| Función | Fórmula \(\varphi(z)\) | Derivada \(\varphi'(z)\) | Rango | Uso |
|---|---|---|---|---|
| **Escalón** (Heaviside) | \(1\) si \(z \geq 0\); si no, \(0\) | \(0\) en todas partes (indefinida en 0) | \(\{0, 1\}\) | unidad de McCulloch–Pitts, perceptrón; no entrenable por gradiente |
| **Sigmoide** (logística) | \(\sigma(z) = \dfrac{1}{1 + e^{-z}}\) | \(\sigma(z)\,(1 - \sigma(z))\) | \((0, 1)\) | unidad de salida para clasificación binaria; también unidades ocultas (lo que usa <code>nnet</code>) |
| **Tangente hiperbólica** | \(\tanh(z)\) | \(1 - \tanh^2(z)\) | \((-1, 1)\) | unidades ocultas; centrada en cero, lo que facilita el entrenamiento en comparación con la sigmoide |
| **ReLU** (unidad lineal rectificada) | \(\max(0, z)\) | \(1\) si \(z > 0\); si no, \(0\) | \([0, \infty)\) | unidad oculta por defecto desde alrededor de 2011; barata, y su derivada no se desvanece para \(z > 0\) |
| **Softmax** | \(\dfrac{e^{z_k}}{\sum_j e^{z_j}}\) para cada salida \(k\) | (una matriz; ver Sección 8.2) | cada una en \((0,1)\), suma 1 | capa de salida para clasificación en \(K\) clases |

La columna de la derivada es tan importante como la de la función, porque el entrenamiento (Sección 8) multiplica derivadas de activaciones a lo largo del camino que va desde la salida hasta cada peso. La derivada de la sigmoide tiene un máximo de \(0.25\) en \(z = 0\) y es prácticamente cero para \(|z| > 5\): una unidad cuya preactivación es grande en magnitud está <em>saturada</em>, y el gradiente que pasa por ella se desvanece; ese es el mecanismo detrás del problema del gradiente que se desvanece de la Sección 9.6. La ReLU, introducida en las redes profundas por Nair y Hinton,<sup class="c"><a href="#r26">[26]</a></sup> tiene derivada exactamente igual a 1 para toda entrada positiva, de modo que los gradientes atraviesan las unidades activas sin disminuir; su precio es que una unidad con \(z < 0\) para todos los casos de entrenamiento tiene gradiente cero y nunca se recupera (una unidad "muerta"). La función escalón tiene derivada cero en todas partes, por eso el perceptrón (Sección 7) necesita una regla de aprendizaje que no sea descenso de gradiente, y por eso la sigmoide la reemplazó en cuanto se quiso entrenar con gradientes. El paquete `nnet` de R usa unidades ocultas logísticas; la ReLU aparece en la red programada desde cero de la Sección 8.6.



In [ ]:
# Las cuatro funciones de activación escalares de la Sección 6.2 y sus derivadas.
z <- seq(-6, 6, length.out = 601)
acts <- list(step = list(f = as.numeric(z >= 0), d = rep(0, length(z))),
             sigmoid = list(f = sigmoid(z), d = sigmoid(z) * (1 - sigmoid(z))),
             tanh = list(f = tanh(z), d = 1 - tanh(z)^2),
             ReLU = list(f = pmax(z, 0), d = as.numeric(z > 0)))
par(mfrow = c(2, 4), mar = c(3.4, 3.4, 2, 1))
for (nm in names(acts)) { plot(z, acts[[nm]]$f, type = "l", col = BLUE, lwd = 1.6, main = nm, xlab = "", ylab = if (nm == "step") "phi(z)" else ""); abline(h = 0, v = 0, col = GREY, lwd = 0.5) }
for (nm in names(acts)) { plot(z, acts[[nm]]$d, type = "l", col = ORANGE, lwd = 1.6, main = "", xlab = "z", ylab = if (nm == "step") "phi'(z)" else "", ylim = c(-0.1, 1.1)); abline(h = 0, v = 0, col = GREY, lwd = 0.5) }

# Softmax convierte K logits en una distribución de probabilidad sobre K clases.
softmax <- function(v) { e <- exp(v - max(v)); e / sum(e) }
logits <- c(2.0, 1.0, 0.1)
cat("logits", logits, "-> softmax", round(softmax(logits), 3), "| sum =", round(sum(softmax(logits)), 6), "\n")
cat("logits + 5 -> softmax", round(softmax(logits + 5), 3), "(adding a constant changes nothing)\n")
cat("logits x 3 -> softmax", round(softmax(logits * 3), 3), "(scaling sharpens the distribution)\n")



<p class="caption"><b>Figura 5.</b> Fila superior: las funciones de activación escalón, sigmoide, tanh y ReLU. Fila inferior: sus derivadas. El escalón tiene derivada cero en todas partes, de modo que ningún gradiente puede fluir a través de él. La derivada de la sigmoide alcanza su máximo en 0.25 y es despreciable más allá de \(|z| \approx 5\) (saturación); la de tanh alcanza su máximo en 1, pero se satura de manera similar. La derivada de la ReLU es 1 para toda entrada positiva —la propiedad que hizo entrenables las redes profundas— y 0 para las entradas negativas. Bajo la figura, la softmax de tres logits: las salidas son positivas, suman uno, no cambian al sumar una constante a todos los logits y se vuelven más marcadas cuando los logits se escalan hacia arriba. Con \(K = 2\) clases, la softmax se reduce a la sigmoide de la diferencia de los dos logits.</p>

### 6.3 Ejemplo resuelto a mano: una neurona calcula AND y OR, y no puede calcular XOR

Toma una unidad de McCulloch–Pitts con dos entradas binarias y activación escalón, \(a = \mathbb{1}[w_1 x_1 + w_2 x_2 + b \geq 0]\). Con pesos \(w_1 = w_2 = 1\) y sesgo \(b = -1.5\), la unidad calcula el **AND** lógico; con los mismos pesos y \(b = -0.5\), calcula **OR**:

| \(x_1\) | \(x_2\) | \(z_{\text{AND}} = x_1 + x_2 - 1.5\) | AND | \(z_{\text{OR}} = x_1 + x_2 - 0.5\) | OR | XOR (objetivo) |
|---|---|---|---|---|---|---|
| 0 | 0 | \(-1.5\) | 0 | \(-0.5\) | 0 | 0 |
| 0 | 1 | \(-0.5\) | 0 | \(0.5\) | 1 | 1 |
| 1 | 0 | \(-0.5\) | 0 | \(0.5\) | 1 | 1 |
| 1 | 1 | \(0.5\) | 1 | \(1.5\) | 1 | 0 |

Geométricamente, la frontera \(z = 0\) es la recta \(x_1 + x_2 = 1.5\) para AND —separa la esquina \((1,1)\) de las otras tres— y la recta \(x_1 + x_2 = 0.5\) para OR, que separa \((0,0)\) de las otras tres. NOT se calcula con una entrada con \(w = -1\), \(b = 0.5\); NAND, con \(w_1 = w_2 = -1\), \(b = 1.5\).

**Por qué ninguna unidad individual calcula XOR.** Supongamos que existieran pesos \(w_1, w_2, b\) con \(a = 1\) exactamente para \((0,1)\) y \((1,0)\). Los cuatro casos exigen

<div class="eq">\[ b < 0 \quad (0,0 \mapsto 0), \qquad w_2 + b \geq 0 \quad (0,1 \mapsto 1), \qquad w_1 + b \geq 0 \quad (1,0 \mapsto 1), \qquad w_1 + w_2 + b < 0 \quad (1,1 \mapsto 0). \]</div>

Sumar la segunda y la tercera desigualdad da \(w_1 + w_2 + 2b \geq 0\), y restar la primera (\(-b > 0\)) da \(w_1 + w_2 + b > 0\), lo que contradice la cuarta. No existen tales pesos. En la imagen, XOR le pide a una sola recta que deje las dos esquinas diagonales \((0,1)\) y \((1,0)\) a un lado y \((0,0)\) y \((1,1)\) al otro, y ninguna recta lo logra: las dos clases no son <strong>linealmente separables</strong>. Esta es la observación que Minsky y Papert convirtieron en una teoría general de lo que los perceptrones de una sola capa no pueden hacer,<sup class="c"><a href="#r3">[3]</a></sup> y la razón por la que las capas ocultas son necesarias: XOR es \((x_1 \text{ OR } x_2) \text{ AND NOT } (x_1 \text{ AND } x_2)\), una composición de tres funciones linealmente separables, que una red con una capa oculta de dos unidades calcula exactamente (la Sección 8.5 entrena una). El bloque de código confirma la tabla y busca exhaustivamente, en una grilla, pesos para XOR.



In [ ]:
# Una unidad de McCulloch-Pitts que calcula AND, OR y NAND, y una búsqueda exhaustiva que muestra que ningún peso calcula XOR.
unit <- function(X, w, b) as.integer(X %*% w + b >= 0)
Xb <- rbind(c(0, 0), c(0, 1), c(1, 0), c(1, 1)); xor_target <- c(0, 1, 1, 0)
truth <- data.frame(x1 = Xb[, 1], x2 = Xb[, 2],
                    AND = unit(Xb, c(1, 1), -1.5), OR = unit(Xb, c(1, 1), -0.5),
                    NAND = unit(Xb, c(-1, -1), 1.5), XOR_target = xor_target)
print(truth, row.names = FALSE)
grid <- seq(-3, 3, length.out = 121); best <- 0
for (w1 in grid) for (w2 in grid) {
  z <- Xb %*% c(w1, w2)                                        # preactivación sin sesgo, longitud 4
  preds <- outer(grid, as.vector(z), function(b, zz) as.integer(zz + b >= 0))   # filas = sesgos candidatos
  best <- max(best, max(rowSums(sweep(preds, 2, xor_target, `==`))))
}
cat(sprintf("\nXOR: best number of correct cases over all %d weight triples = %d of 4\n", length(grid)^3, best))
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
for (cfg in list(list("AND", truth$AND, 1.5), list("OR", truth$OR, 0.5), list("XOR", xor_target, NA))) {
  nm <- cfg[[1]]; tgt <- cfg[[2]]; ln <- cfg[[3]]
  plot(Xb[, 1], Xb[, 2], pch = ifelse(tgt == 1, 15, 19), col = ifelse(tgt == 1, BLUE, ORANGE), cex = 2.2,
       xlim = c(-0.4, 1.4), ylim = c(-0.4, 1.4), xlab = "x1", ylab = "x2", main = nm)
  if (!is.na(ln)) abline(a = ln, b = -1, lty = 2) else text(0.5, 0.5, "no line\nseparates", col = GREY, cex = 0.9)
}



<p class="caption"><b>Figura 6.</b> Los cuatro patrones de entrada de dos variables binarias, coloreados según la salida de AND (izquierda), OR (centro) y XOR (derecha). Para AND y OR, la recta discontinua \(z = 0\) de la unidad resuelta a mano separa las dos salidas. Para XOR, la búsqueda exhaustiva sobre más de 1.7 millones de tríos de pesos acierta como máximo 3 de 4 casos, lo que confirma la demostración algebraica: las dos clases ocupan las diagonales del cuadrado y ninguna recta las separa.</p>

<div class="box key"><h4>Ejercicio 2</h4><p>Encuentra a mano pesos y sesgo para NOR y para la función "\(x_1\) AND NOT \(x_2\)", compruébalos con <code>unit()</code> y dibuja sus fronteras. Luego calcula XOR en papel como una red de dos pasos: unidad oculta \(h_1 = \text{OR}(x_1, x_2)\), unidad oculta \(h_2 = \text{NAND}(x_1, x_2)\), salida \(= \text{AND}(h_1, h_2)\). Escribe la tabla de \(h_1, h_2\) y de la salida para las cuatro entradas. La Sección 8.5 aprenderá estos pesos a partir de los datos en lugar de diseñarlos.</p></div>

## 7. El perceptrón simple

### 7.1 La máquina de Rosenblatt

El perceptrón de Rosenblatt de 1958 fue la primera máquina que aprendió a clasificar a partir de ejemplos.<sup class="c"><a href="#r2">[2]</a></sup> El Mark I Perceptron tomaba su entrada de una grilla de 20 × 20 fotocélulas (<em>unidades sensoriales</em>), las conectaba mediante un cableado aleatorio fijo con <em>unidades de asociación</em>, y estas, a través de pesos ajustables —potenciómetros accionados por motores—, con <em>unidades de respuesta</em>; solo el último conjunto de conexiones aprendía. En términos modernos, el perceptrón es una red de una sola capa de unidades de umbral, entrenada con la regla de la subsección siguiente; el "perceptrón simple" del programa del curso es una de esas unidades, que calcula \(\hat{y} = \text{sign}(\mathbf{w}^\top \mathbf{x} + b)\) con las clases codificadas como \(y \in \{-1, +1\}\).

### 7.2 La regla de aprendizaje y el teorema de convergencia

La regla es una versión supervisada de la de Hebb: ajustar los pesos solo cuando la unidad se equivoca, y en la dirección que la habría hecho acertar. Para un ejemplo de entrenamiento \((\mathbf{x}, y)\) con predicción \(\hat{y}\),

<div class="eq">\[ \text{if } \hat{y} \neq y: \qquad \mathbf{w} \leftarrow \mathbf{w} + \eta\, y\, \mathbf{x}, \qquad b \leftarrow b + \eta\, y, \]
<span class="where">donde \(\eta > 0\) es la tasa de aprendizaje. Si la predicción es correcta, nada cambia.</span></div>

La regla tiene una derivación transparente. Un error en un ejemplo positivo (\(y = +1\), pero \(z = \mathbf{w}^\top\mathbf{x} + b < 0\)) significa que \(z\) era demasiado pequeño; después de la actualización, la preactivación del mismo ejemplo pasa a ser \(z + \eta(\lVert \mathbf{x} \rVert^2 + 1)\), que es mayor, de modo que el ejemplo se ha desplazado hacia el lado correcto. Un error en un ejemplo negativo hace bajar \(z\) en la misma cantidad. Cada actualización es un paso de tamaño \(\eta\) en la dirección de \(y\mathbf{x}\), que es el gradiente negativo de la <em>pérdida del perceptrón</em> \(\max(0, -y z)\): la regla es un descenso de gradiente estocástico sobre una pérdida que vale cero para los ejemplos bien clasificados y crece linealmente con la violación para los mal clasificados. El algoritmo recorre cíclicamente la muestra de entrenamiento (una pasada es una <strong>época</strong>) hasta que una época no produce ninguna actualización.

¿Se detiene? Rosenblatt afirmó que sí; Novikoff dio la demostración en 1962.<sup class="c"><a href="#r23">[23]</a></sup> <strong>Teorema de convergencia del perceptrón.</strong> Si la muestra de entrenamiento es linealmente separable con margen \(\gamma\) —es decir, algún vector de pesos de norma unitaria \(\mathbf{w}^*\) (con el sesgo incorporado) cumple \(y_i\, \mathbf{w}^{*\top} \mathbf{x}_i \geq \gamma > 0\) para todo \(i\)— y todo ejemplo cumple \(\lVert \mathbf{x}_i \rVert \leq R\), entonces la regla del perceptrón, partiendo de \(\mathbf{w} = \mathbf{0}\), comete como máximo \((R/\gamma)^2\) errores antes de clasificar correctamente todos los ejemplos, cualquiera que sea el orden de presentación y para cualquier \(\eta\). La cota no depende ni del número de ejemplos ni de la dimensión. La premisa del teorema es también su limitación: si la muestra no es separable, el algoritmo nunca se detiene (Sección 7.4).

### 7.3 Ejemplo resuelto a mano: una época del perceptrón sobre AND

Codifica el problema AND con \(y \in \{-1, +1\}\): las entradas \((0,0), (0,1), (1,0)\) tienen \(y = -1\) y \((1,1)\) tiene \(y = +1\). Toma \(\eta = 1\), parte de \(\mathbf{w} = (0, 0)\), \(b = 0\), predice \(\hat{y} = +1\) cuando \(z \geq 0\) y \(-1\) en caso contrario, y presenta los cuatro ejemplos en el orden anterior. La primera época:

| paso | \(\mathbf{x}\) | \(y\) | \(z = \mathbf{w}^\top\mathbf{x} + b\) | \(\hat{y}\) | ¿error? | actualización | \(\mathbf{w}\) después | \(b\) después |
|---|---|---|---|---|---|---|---|---|
| 1 | \((0,0)\) | \(-1\) | \(0\) | \(+1\) | sí | \(b \mathrel{+}= -1\) | \((0, 0)\) | \(-1\) |
| 2 | \((0,1)\) | \(-1\) | \(-1\) | \(-1\) | no | — | \((0, 0)\) | \(-1\) |
| 3 | \((1,0)\) | \(-1\) | \(-1\) | \(-1\) | no | — | \((0, 0)\) | \(-1\) |
| 4 | \((1,1)\) | \(+1\) | \(-1\) | \(-1\) | sí | \(\mathbf{w} \mathrel{+}= (1,1)\), \(b \mathrel{+}= 1\) | \((1, 1)\) | \(0\) |

Después de una época, \(\mathbf{w} = (1, 1)\) y \(b = 0\): la recta \(x_1 + x_2 = 0\) pasa por el origen, lo que sigue siendo incorrecto para \((0,0)\) (\(z = 0 \Rightarrow \hat{y} = +1\)). La segunda época comienza con un error en \((0,0)\) (el sesgo pasa a \(-1\)), luego otro en \((0,1)\) (\(z = 0\); \(\mathbf{w} \leftarrow (1, 0)\), \(b \leftarrow -2\)), y así sucesivamente. El bloque de código siguiente ejecuta la misma aritmética hasta la convergencia; debería reproducir exactamente las cuatro filas anteriores y terminar en una recta separadora.



In [ ]:
# La regla de aprendizaje del perceptrón en R base, sobre el problema AND, registrando cada paso (compárala con la tabla hecha a mano).
perceptron_fit <- function(X, y, eta = 1, max_epochs = 20, log = FALSE) {
  w <- rep(0, ncol(X)); b <- 0; mistakes <- integer(0); rows <- list()
  for (epoch in 1:max_epochs) {
    m <- 0
    for (i in seq_len(nrow(X))) {
      z <- sum(w * X[i, ]) + b; yhat <- if (z >= 0) 1 else -1
      miss <- yhat != y[i]
      if (miss) { w <- w + eta * y[i] * X[i, ]; b <- b + eta * y[i]; m <- m + 1 }
      if (log) rows[[length(rows) + 1]] <- data.frame(epoch, x1 = X[i, 1], x2 = X[i, 2], y = y[i],
                 z = z, yhat = yhat, mistake = miss, w1 = w[1], w2 = w[2], b = b)
    }
    mistakes <- c(mistakes, m); if (m == 0) break
  }
  list(w = w, b = b, mistakes = mistakes, log = if (log) do.call(rbind, rows) else NULL)
}
X_and <- Xb; y_and <- c(-1, -1, -1, 1)
fit_and <- perceptron_fit(X_and, y_and, log = TRUE)
print(fit_and$log[fit_and$log$epoch == 1, ], row.names = FALSE)                 # época 1: las filas seguidas a mano
cat(sprintf("\nmistakes per epoch: %s -> converged after %d epochs, %d updates total\n",
            paste(fit_and$mistakes, collapse = " "), length(fit_and$mistakes), sum(fit_and$mistakes)))
cat(sprintf("final w = (%g, %g), b = %g\n", fit_and$w[1], fit_and$w[2], fit_and$b))



Las cuatro primeras filas coinciden línea por línea con la tabla hecha a mano. El algoritmo necesita seis épocas —cinco con errores y una sexta que confirma que ya no hay ninguno— y once actualizaciones en total, y termina con \(\mathbf{w} = (2, 1)\), \(b = -3\): la recta \(2x_1 + x_2 = 3\), que pasa exactamente por \((1,1)\) (clasificado como \(+1\) porque \(z = 0 \geq 0\)) y deja las otras tres esquinas en el lado negativo. Es <em>una</em> recta separadora, no la "mejor"; el perceptrón se detiene en la primera solución que encuentra, y un orden de presentación distinto produciría una recta distinta.

<div class="box warn"><h4>Error frecuente</h4><p>"El perceptrón minimiza el número de errores". No lo hace: se detiene en el primer vector de pesos con cero errores de entrenamiento, si existe, y en caso contrario nunca se detiene. La máquina de vectores de soporte de la Unidad 3 resuelve el problema que el perceptrón deja abierto —<em>cuál</em> recta separadora— eligiendo la de mayor margen, y la regresión logística (Unidad 2) y el Adaline de la Sección 7.5 minimizan pérdidas suaves que están definidas tanto si los datos son separables como si no.</p></div>

### 7.4 El perceptrón con datos reales: cáncer de mama, y el fracaso con XOR

El bloque de código siguiente aplica la regla a las dos características estandarizadas de la Sección 3.3 para los 683 casos y dibuja la recta de decisión después de algunas épocas seleccionadas. Las dos clases se superponen, de modo que la muestra no es linealmente separable y el teorema no se aplica; la figura muestra lo que ocurre entonces.



In [ ]:
# El perceptrón sobre dos características (estandarizadas) de cáncer de mama. Las clases se superponen: la frontera no deja de moverse.
y_pm <- ifelse(y_mal == 1, 1, -1)                                    # +1 = maligno, -1 = benigno
snapshots <- c(1, 2, 3, 5, 10, 20, 50, 100); w <- c(0, 0); b <- 0; states <- list()
for (epoch in 1:100) {
  for (i in seq_len(nrow(Z2))) if (y_pm[i] * (sum(w * Z2[i, ]) + b) <= 0) { w <- w + y_pm[i] * Z2[i, ]; b <- b + y_pm[i] }
  if (epoch %in% snapshots) states[[as.character(epoch)]] <- list(w = w, b = b,
      acc = mean(ifelse(Z2 %*% w + b >= 0, 1, -1) == y_pm))
}
cat("training accuracy after epoch:",
    paste(sprintf("%d:%.3f", snapshots, sapply(as.character(snapshots), function(e) states[[e]]$acc)), collapse = "  "), "\n")
par(mfrow = c(2, 4), mar = c(3.6, 3.6, 2, 1)); xs <- c(-2.5, 4.5)
for (e in snapshots) {
  st <- states[[as.character(e)]]
  plot(Z2[, 1], Z2[, 2], col = ifelse(y_pm == 1, ORANGE, BLUE), pch = 19, cex = 0.3, xlim = c(-2.5, 4.5),
       ylim = c(-2, 4.5), xlab = "cell size (std.)", ylab = "bare nuclei (std.)", main = sprintf("epoch %d: acc %.3f", e, st$acc))
  lines(xs, -(st$w[1] * xs + st$b) / st$w[2], lwd = 1.3)
}



<p class="caption"><b>Figura 7.</b> La recta de decisión del perceptrón después de 1, 2, 3, 5, 10, 20, 50 y 100 épocas en el problema de cáncer de mama con dos características. La exactitud de entrenamiento ya es alta después de la primera época y luego <em>fluctúa</em> en lugar de mejorar, porque cada época contiene casos mal clasificados que mueven la recta; el algoritmo nunca termina, y la recta de la época 100 no es sistemáticamente mejor que la de la época 5. Con datos no separables, los pesos finales del perceptrón dependen de cuándo se decida detenerlo, una elección arbitraria que los métodos de la Sección 7.5 y de la Unidad 2 eliminan al minimizar una pérdida en lugar de perseguir cero errores.</p>



In [ ]:
# Errores por época: el perceptrón converge en AND (separable) y nunca en XOR (no separable).
X_xor <- Xb; y_xor <- c(-1, 1, 1, -1)
hist_and <- perceptron_fit(X_and, y_and, max_epochs = 30)$mistakes
hist_xor <- perceptron_fit(X_xor, y_xor, max_epochs = 30)$mistakes
cat("AND: mistakes per epoch", hist_and, "\n")
cat("XOR: mistakes per epoch", hist_xor, sprintf("-> minimum %d per epoch, never 0\n", min(hist_xor)))
plot(seq_along(hist_and), hist_and, type = "o", pch = 19, col = GREEN, ylim = c(0, 4), xlab = "epoch",
     ylab = "mistakes in the epoch", main = "Perceptron convergence theorem in action")
lines(seq_along(hist_xor), hist_xor, type = "o", pch = 15, col = ORANGE)
legend("topright", c("AND (separable)", "XOR (not separable)"), col = c(GREEN, ORANGE), lwd = 1, pch = c(19, 15), bty = "n", cex = 0.8)



<p class="caption"><b>Figura 8.</b> Errores por época de la regla del perceptrón en AND y en XOR. En AND el conteo llega a cero en la sexta época y el algoritmo se detiene, tal como garantiza el teorema de convergencia. En XOR el conteo nunca llega a cero dentro de las 30 épocas permitidas —ni lo haría en 30 millones—, porque no existe ninguna recta separadora; los pesos recorren cíclicamente un conjunto finito de estados para siempre. Este es el experimento detrás de la crítica de Minsky y Papert.<sup class="c"><a href="#r3">[3]</a></sup></p>

### 7.5 Adaline, la regla delta y el vínculo con el descenso de gradiente

Dos años después de Rosenblatt, Widrow y Hoff propusieron una unidad que se diferencia del perceptrón en un solo aspecto, pero de grandes consecuencias.<sup class="c"><a href="#r24">[24]</a></sup> El <strong>Adaline</strong> (adaptive linear neuron, neurona lineal adaptativa) compara el objetivo no con la salida umbralizada \(\hat{y} = \text{sign}(z)\), sino con la propia salida <em>lineal</em> \(z = \mathbf{w}^\top\mathbf{x} + b\), y minimiza el error cuadrático \(L = \tfrac{1}{2}(y - z)^2\) mediante descenso de gradiente. Como \(\partial L / \partial \mathbf{w} = -(y - z)\,\mathbf{x}\), la actualización es

<div class="eq">\[ \mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - z)\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - z), \]
<span class="where">la <strong>regla delta</strong> (regla de Widrow–Hoff, regla LMS), llamada así por el error \(\delta = y - z\).</span></div>

Tres cosas cambian respecto de la regla del perceptrón: la actualización se aplica a <em>todos</em> los ejemplos, escalada según cuán equivocada esté la salida lineal; la pérdida es suave y convexa, de modo que el descenso de gradiente converge a su mínimo único tanto si los datos son separables como si no; y ahora el tamaño del paso \(\eta\) importa: si es demasiado grande diverge y si es demasiado pequeño avanza a paso de tortuga (Sección 9.1). Si se reemplaza el error cuadrático por la entropía cruzada y la salida lineal por una sigmoide, el Adaline se convierte en la regresión logística de la Unidad 2; la Sección 8 generaliza su "error por entrada" a unidades ocultas cuyo error no es directamente observable. El bloque de código aplica el Adaline a las mismas dos características con tres tasas de aprendizaje.



In [ ]:
# Adaline (regla delta): descenso de gradiente de lote completo sobre el error cuadrático, tres tasas de aprendizaje.
adaline_fit <- function(X, y, eta, epochs = 60) {
  w <- rep(0, ncol(X)); b <- 0; losses <- numeric(epochs)
  for (e in 1:epochs) {
    z <- X %*% w + b; err <- y - z
    w <- w + eta * t(X) %*% err / length(y); b <- b + eta * mean(err)
    losses[e] <- 0.5 * mean((y - (X %*% w + b))^2)
  }
  list(w = w, b = b, losses = losses)
}
rho <- cor(Z2)[1, 2]
cat(sprintf("correlation of the two features = %.3f -> divergence for eta > 2/(1+rho) = %.2f\n", rho, 2 / (1 + rho)))
plot(NA, xlim = c(1, 60), ylim = c(0.1, 1e3), log = "y", xlab = "epoch", ylab = "squared-error loss (log scale)",
     main = "Adaline: crawling, converging and diverging")
for (cfg in list(c(0.01, 1), c(0.3, 2), c(1.25, 3))) {
  eta <- cfg[1]; fit <- adaline_fit(Z2, y_pm, eta)
  acc <- mean(ifelse(Z2 %*% fit$w + fit$b >= 0, 1, -1) == y_pm)
  cat(sprintf("eta = %-5g: loss after 60 epochs = %10.3g, training accuracy = %.3f\n", eta, fit$losses[60], acc))
  lines(1:60, fit$losses, col = c(GREY, BLUE, ORANGE)[cfg[2]], lwd = 1.4)
}
legend("topright", c("eta = 0.01", "eta = 0.3", "eta = 1.25"), col = c(GREY, BLUE, ORANGE), lwd = 1.4, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 9.</b> Pérdida por época del Adaline para tres tasas de aprendizaje. Con \(\eta = 0.01\) la pérdida disminuye, pero no ha convergido después de 60 épocas; con \(\eta = 0.3\) alcanza su mínimo en unas veinte; con \(\eta = 1.25\), justo por encima del umbral de estabilidad, cada paso sobrepasa el mínimo en más de lo que gana y la pérdida <em>crece</em>: divergencia. Para una pérdida cuadrática convexa el umbral es exacto: el descenso de gradiente con lote completo diverge cuando \(\eta\) supera \(2/\lambda_{\max}\), donde \(\lambda_{\max}\) es el mayor valor propio de la matriz de covarianza de las características; para dos características estandarizadas con correlación \(\rho\), \(\lambda_{\max} = 1 + \rho\), lo que da el umbral impreso sobre la figura. El Adaline alcanza una exactitud cercana a la del perceptrón, pero mediante un procedimiento que se detiene.</p>

<div class="box bridge"><h4>Puente con la Unidad 2</h4><p>El Adaline es exactamente la regresión lineal de la Unidad 2, ajustada por descenso de gradiente en lugar de mediante las ecuaciones normales, con la clase codificada como \(\pm 1\). La regla delta, el paso de gradiente de la Unidad 2 para la regresión logística y la retropropagación de la sección siguiente son la misma instrucción —<em>mover cada peso en proporción al error multiplicado por la entrada que ese peso multiplica</em>— aplicada a modelos de profundidad creciente.</p></div>

## 8. El perceptrón multicapa

### 8.1 Arquitectura y propagación hacia adelante

Un <strong>perceptrón multicapa</strong> es una red prealimentada con al menos una capa oculta, en la que cada capa está totalmente conectada con la siguiente. Pese al nombre, sus unidades no son perceptrones en el sentido de Rosenblatt: usan activaciones diferenciables (sigmoide, tanh, ReLU) en lugar del escalón, precisamente para que la red pueda entrenarse con descenso de gradiente. El cálculo de la capa \(l\), para los \(n\) casos de entrenamiento a la vez, es un producto de matrices seguido de una activación elemento a elemento:

<div class="eq">\[ \mathbf{Z}^{(l)} = \mathbf{A}^{(l-1)} \mathbf{W}^{(l)} + \mathbf{b}^{(l)}, \qquad \mathbf{A}^{(l)} = \varphi^{(l)}\!\left(\mathbf{Z}^{(l)}\right), \qquad \mathbf{A}^{(0)} = \mathbf{X}, \]
<span class="where">donde \(\mathbf{X}\) es la matriz de datos de \(n \times n_0\), \(\mathbf{W}^{(l)}\) es de \(n_{l-1} \times n_l\), \(\mathbf{b}^{(l)}\) es una fila de \(n_l\) sesgos que se suma a cada fila (reciclada a lo largo de las filas), y \(\mathbf{Z}^{(l)}\) y \(\mathbf{A}^{(l)}\) son de \(n \times n_l\). La salida \(\hat{\mathbf{Y}} = \mathbf{A}^{(L)}\) es de \(n \times n_L\). Esto es la <strong>propagación hacia adelante</strong> (forward propagation).</span></div>

Para la red 9-16-1 con los datos de cáncer de mama: \(\mathbf{X}\) es de \(683 \times 9\), \(\mathbf{W}^{(1)}\) es de \(9 \times 16\), \(\mathbf{A}^{(1)}\) es de \(683 \times 16\) (las dieciséis características aprendidas de cada caso), \(\mathbf{W}^{(2)}\) es de \(16 \times 1\) y \(\hat{\mathbf{y}}\) es una columna de 683 probabilidades.

### 8.2 Funciones de pérdida

El entrenamiento necesita un número, que se minimizará, que diga cuán equivocada está la red en la muestra de entrenamiento. Para la <strong>regresión</strong> con una unidad de salida lineal, la elección natural es el <strong>error cuadrático medio</strong>, \(L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2\), como en la Unidad 2. Para la <strong>clasificación binaria</strong> con una salida sigmoide interpretada como \(\hat{y} = P(y = 1 \mid \mathbf{x})\), la pérdida es la <strong>entropía cruzada binaria</strong> (log-verosimilitud negativa del modelo de Bernoulli),

<div class="eq">\[ L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \ln \hat{y}_i + (1 - y_i) \ln (1 - \hat{y}_i) \right], \]</div>

y para \(K\) clases con salida softmax y objetivos codificados one-hot \(y_{ik}\), la <strong>entropía cruzada categórica</strong> \(L = -\frac{1}{n}\sum_i \sum_k y_{ik} \ln \hat{y}_{ik}\). Para clasificación se prefiere la entropía cruzada al error cuadrático por una razón que se hace visible en la derivación siguiente: con una salida sigmoide y entropía cruzada, la señal de error en la salida es simplemente \(\hat{y} - y\), mientras que con error cuadrático es \((\hat{y} - y)\,\sigma'(z)\), que se desvanece cuando la unidad de salida está saturada; una red que se equivoca con mucha confianza aprendería entonces casi nada de su error. La misma simplificación vale para softmax con entropía cruzada categórica: la derivada de la pérdida respecto de los logits es \(\hat{\mathbf{y}} - \mathbf{y}\). El seguimiento a mano de la Sección 8.4 usa el error cuadrático para que todos los factores queden a la vista; la implementación general de la Sección 8.6 usa entropía cruzada.

### 8.3 Retropropagación: la regla de la cadena, organizada

La minimización se hace por descenso de gradiente, \(\theta \leftarrow \theta - \eta\, \partial L / \partial \theta\) para cada peso y sesgo, de modo que todo el problema consiste en calcular las derivadas parciales. Para una red con capas ocultas ese es un problema de contabilidad, y la <strong>retropropagación</strong> (backpropagation) es esa contabilidad: una organización eficiente de la regla de la cadena, descubierta varias veces y llevada a la atención del campo por Rumelhart, Hinton y Williams en 1986.<sup class="c"><a href="#r4">[4]</a></sup> Su idea es definir, para cada unidad, la derivada de la pérdida respecto de la <em>preactivación</em> de esa unidad, \(\delta = \partial L / \partial z\), y notar que el \(\delta\) de una capa puede calcularse a partir del \(\delta\) de la capa siguiente; así, las derivadas se calculan hacia atrás, desde la salida hasta la entrada, en una sola pasada que cuesta aproximadamente lo mismo que la pasada hacia adelante.

Toma una red 2-2-1 con activaciones sigmoides en todas partes y la pérdida de error cuadrático para un ejemplo, \(L = \tfrac{1}{2}(a^{(2)} - y)^2\). Hacia adelante: \(z_j^{(1)} = \sum_i W^{(1)}_{ij} x_i + b^{(1)}_j\), \(a_j^{(1)} = \sigma(z_j^{(1)})\) para \(j = 1, 2\); \(z^{(2)} = \sum_j W^{(2)}_j a_j^{(1)} + b^{(2)}\), \(a^{(2)} = \sigma(z^{(2)})\). Hacia atrás, por la regla de la cadena:

**Unidad de salida.** \(\delta^{(2)} = \dfrac{\partial L}{\partial z^{(2)}} = (a^{(2)} - y)\; a^{(2)} (1 - a^{(2)})\). Luego, \(\dfrac{\partial L}{\partial W^{(2)}_j} = \delta^{(2)} a_j^{(1)}\) y \(\dfrac{\partial L}{\partial b^{(2)}} = \delta^{(2)}\): "error por entrada", la regla delta.

**Unidades ocultas.** La unidad oculta \(j\) afecta a \(L\) solo a través de \(z^{(2)}\), de modo que \(\delta^{(1)}_j = \delta^{(2)}\, W^{(2)}_j\; a_j^{(1)}(1 - a_j^{(1)})\): el error de salida se envía <em>de vuelta a lo largo del peso</em> que conecta las dos unidades y se escala por la derivada de la activación oculta. Luego, \(\dfrac{\partial L}{\partial W^{(1)}_{ij}} = \delta^{(1)}_j x_i\) y \(\dfrac{\partial L}{\partial b^{(1)}_j} = \delta^{(1)}_j\).

El patrón se generaliza a cualquier profundidad. En forma matricial, para un minilote, con \(\odot\) el producto elemento a elemento:

<div class="eq">\[ \boldsymbol{\Delta}^{(L)} = \frac{\partial L}{\partial \mathbf{Z}^{(L)}}, \quad \boldsymbol{\Delta}^{(l)} = \left(\boldsymbol{\Delta}^{(l+1)} \mathbf{W}^{(l+1)\top}\right) \odot \varphi'\!\left(\mathbf{Z}^{(l)}\right), \quad \frac{\partial L}{\partial \mathbf{W}^{(l)}} = \mathbf{A}^{(l-1)\top} \boldsymbol{\Delta}^{(l)}, \quad \frac{\partial L}{\partial \mathbf{b}^{(l)}} = \mathbf{1}^\top \boldsymbol{\Delta}^{(l)}. \]
<span class="where">Para una salida sigmoide con entropía cruzada, o una salida softmax con entropía cruzada categórica, \(\boldsymbol{\Delta}^{(L)} = (\hat{\mathbf{Y}} - \mathbf{Y})/n\). Estas cuatro líneas son la totalidad de la función <code>backward</code> de la Sección 8.6.</span></div>

### 8.4 Ejemplo resuelto a mano: un paso de entrenamiento de la red 2-2-1

Toma el ejemplo de XOR \(\mathbf{x} = (1, 1)\) con objetivo \(y = 0\), tasa de aprendizaje \(\eta = 0.5\) y los pesos

<div class="eq">\[ \mathbf{W}^{(1)} = \begin{pmatrix} 0.5 & -0.5 \\ 0.3 & 0.8 \end{pmatrix}, \quad \mathbf{b}^{(1)} = (0.1, -0.2), \quad \mathbf{W}^{(2)} = (0.6, -0.4), \quad b^{(2)} = 0.2, \]</div>

con la convención de que <em>la fila \(j\) de \(\mathbf{W}^{(1)}\) contiene los pesos que entran a la unidad oculta \(j\)</em>, de modo que \(\mathbf{z}^{(1)} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\). (La forma matricial de la Sección 8.1 usa la traspuesta, \(\mathbf{X}\mathbf{W}\), porque allí los casos son filas; los libros de texto difieren, y la única regla es usar una convención de forma consistente).

**Pasada hacia adelante.** \(z^{(1)} = (0.1,\ 0.9)\), \(a^{(1)} = (\sigma(0.1), \sigma(0.9)) = (0.5250,\ 0.7109)\); \(z^{(2)} = 0.6 \times 0.5250 - 0.4 \times 0.7109 + 0.2 = 0.2306\), \(a^{(2)} = \sigma(0.2306) = 0.5574\); \(L = \tfrac{1}{2}(0.5574)^2 = 0.1553\).

**Pasada hacia atrás.** \(\delta^{(2)} = (a^{(2)} - y)\,a^{(2)}(1 - a^{(2)}) = 0.5574 \times 0.5574 \times 0.4426 = 0.1375\). \(\partial L/\partial W^{(2)} = \delta^{(2)} a^{(1)} = (0.0722,\ 0.0978)\); \(\partial L/\partial b^{(2)} = 0.1375\). \(\delta^{(1)}_1 = 0.1375 \times 0.6 \times 0.5250 \times 0.4750 = 0.0206\); \(\delta^{(1)}_2 = 0.1375 \times (-0.4) \times 0.7109 \times 0.2891 = -0.0113\). \(\partial L/\partial W^{(1)}_{ji} = \delta^{(1)}_j x_i\): ambas entradas valen 1, así que la fila 1 del gradiente es \((0.0206, 0.0206)\) y la fila 2 es \((-0.0113, -0.0113)\); \(\partial L/\partial b^{(1)} = (0.0206, -0.0113)\).

**Actualización** (\(\theta \leftarrow \theta - 0.5\, \partial L/\partial\theta\)). \(W^{(2)} \leftarrow (0.5639,\ -0.4489)\), \(b^{(2)} \leftarrow 0.1312\); \(W^{(1)} \leftarrow \begin{pmatrix} 0.4897 & -0.5103 \\ 0.3057 & 0.8057 \end{pmatrix}\), \(b^{(1)} \leftarrow (0.0897,\ -0.1943)\).

Observa la dirección de cada cambio: la salida fue 0.5574 para un objetivo de 0, así que \(W^{(2)}_1\), que multiplica una activación oculta positiva, disminuyó, y \(W^{(2)}_2\), que ya era negativo, se hizo más negativo; ambos empujan \(z^{(2)}\) hacia abajo. Los cambios en la capa oculta son un orden de magnitud menores que los de la capa de salida: en su camino de vuelta, el error se ha multiplicado por un peso de magnitud \(\leq 0.6\) y por una derivada de la sigmoide \(\leq 0.25\). Una capa más lo reduciría de nuevo; este es, en miniatura, el gradiente que se desvanece de la Sección 9.6. El bloque de código reproduce cada número y luego comprueba el gradiente contra una diferencia finita.



In [ ]:
# Retropropagación a mano, en código: pasada hacia adelante, deltas, gradientes y actualización para el ejemplo 2-2-1.
x_ex <- c(1, 1); y_ex <- 0; eta <- 0.5
W1 <- matrix(c(0.5, -0.5, 0.3, 0.8), nrow = 2, byrow = TRUE); b1 <- c(0.1, -0.2)   # fila j = pesos que ENTRAN a la unidad oculta j
W2 <- c(0.6, -0.4); b2 <- 0.2
z1 <- as.vector(W1 %*% x_ex + b1); a1 <- sigmoid(z1)                                # capa oculta
z2 <- sum(W2 * a1) + b2;           a2 <- sigmoid(z2)                                # unidad de salida
loss <- 0.5 * (a2 - y_ex)^2
cat(sprintf("forward : z1 = (%.4f, %.4f), a1 = (%.4f, %.4f), z2 = %.4f, a2 = %.4f, loss = %.4f\n",
            z1[1], z1[2], a1[1], a1[2], z2, a2, loss))
d2 <- (a2 - y_ex) * a2 * (1 - a2); gW2 <- d2 * a1; gb2 <- d2                        # delta en la salida
d1 <- d2 * W2 * a1 * (1 - a1); gW1 <- outer(d1, x_ex); gb1 <- d1                    # deltas en las unidades ocultas
cat(sprintf("backward: delta2 = %.4f, dL/dW2 = (%.4f, %.4f)\n", d2, gW2[1], gW2[2]))
cat(sprintf("          delta1 = (%.4f, %.4f)\n", d1[1], d1[2]))
cat(sprintf("update  : W2 -> (%.4f, %.4f), b2 -> %.4f\n", (W2 - eta * gW2)[1], (W2 - eta * gW2)[2], b2 - eta * gb2))
cat("          W1 ->\n"); print(round(W1 - eta * gW1, 4))
loss_of <- function(W1_) 0.5 * (sigmoid(sum(W2 * sigmoid(as.vector(W1_ %*% x_ex + b1))) + b2) - y_ex)^2
h <- 1e-6; Wp <- W1; Wm <- W1; Wp[1, 1] <- Wp[1, 1] + h; Wm[1, 1] <- Wm[1, 1] - h
cat(sprintf("check   : dL/dW1[1,1] backprop = %.7f, finite difference = %.7f\n",
            gW1[1, 1], (loss_of(Wp) - loss_of(Wm)) / (2 * h)))



Cada número del seguimiento a mano se reproduce hasta el cuarto decimal, y la derivada por diferencia finita —la pérdida recalculada con un peso desplazado levemente hacia arriba y hacia abajo— coincide con la retropropagada hasta el séptimo decimal. La comprobación por diferencias finitas es demasiado lenta para entrenar (dos pasadas hacia adelante por parámetro), pero es la forma estándar de verificar una implementación, y la Sección 8.7 la aplica a la implementación general.

### 8.5 Entrenar XOR hasta la convergencia

Repetir el paso de la Sección 8.4 sobre los cuatro patrones de XOR, miles de veces, es entrenar. El bloque de código siguiente lo hace con descenso de gradiente de lote completo sobre el error cuadrático y la misma arquitectura 2-2-1, partiendo de pesos aleatorios, y luego muestra la función que la red ha aprendido sobre todo el cuadrado unitario.



In [ ]:
# Entrenamiento de la red sigmoide 2-2-1 sobre XOR con descenso de gradiente de lote completo (pérdida de error cuadrático, como en el seguimiento a mano).
set.seed(0)
X_xor01 <- Xb; y_xor01 <- c(0, 1, 1, 0)
W1 <- matrix(rnorm(4), 2, 2); b1 <- c(0, 0); W2 <- matrix(rnorm(2), 2, 1); b2 <- 0; losses <- numeric(10000)
for (epoch in 1:10000) {
  A1 <- sigmoid(X_xor01 %*% W1 + matrix(b1, 4, 2, byrow = TRUE)); A2 <- sigmoid(A1 %*% W2 + b2)   # hacia adelante
  D2 <- (A2 - y_xor01) * A2 * (1 - A2) / 4                                                         # deltas de salida
  D1 <- (D2 %*% t(W2)) * A1 * (1 - A1)                                                             # deltas ocultos
  W2 <- W2 - t(A1) %*% D2; b2 <- b2 - sum(D2); W1 <- W1 - t(X_xor01) %*% D1; b1 <- b1 - colSums(D1)
  losses[epoch] <- 0.5 * mean((as.vector(A2) - y_xor01)^2)
}
out <- as.vector(sigmoid(sigmoid(X_xor01 %*% W1 + matrix(b1, 4, 2, byrow = TRUE)) %*% W2 + b2))
cat("loss at epochs 1, 1000, 5000, 10000:", round(losses[c(1, 1000, 5000, 10000)], 4), "\n")
cat("network output for (0,0),(0,1),(1,0),(1,1):", round(out, 3), "-> rounded:", as.integer(out >= 0.5), "\n")
gx <- seq(-0.5, 1.5, length.out = 120); gr <- as.matrix(expand.grid(x1 = gx, x2 = gx))
go <- sigmoid(sigmoid(gr %*% W1 + matrix(b1, nrow(gr), 2, byrow = TRUE)) %*% W2 + b2)
par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
plot(losses, type = "l", col = BLUE, log = "x", xlab = "epoch (log scale)", ylab = "squared-error loss", main = "Training loss on XOR")
image(gx, gx, matrix(go, 120, 120), col = hcl.colors(20, "RdYlBu", rev = TRUE), xlab = "x1", ylab = "x2", main = "Learned function")
contour(gx, gx, matrix(go, 120, 120), levels = 0.5, add = TRUE, lwd = 1.4)
points(Xb[y_xor01 == 0, , drop = FALSE], pch = 21, bg = "white", cex = 1.6); points(Xb[y_xor01 == 1, , drop = FALSE], pch = 21, bg = "black", cex = 1.6)



<p class="caption"><b>Figura 10.</b> Izquierda: la pérdida de error cuadrático durante 10,000 épocas de descenso de gradiente sobre XOR. La pérdida permanece en una meseta cercana a 0.125 durante las primeras miles de épocas —la salida es de alrededor de 0.5 para todos los patrones— y luego cae bruscamente una vez que las unidades ocultas se han diferenciado. Derecha: la función aprendida sobre el cuadrado unitario. La unidad de salida asigna una salida baja a la franja que contiene \((0,0)\) y \((1,1)\) (puntos blancos) y una salida alta donde se encuentran \((0,1)\) y \((1,0)\) (puntos negros). La frontera de decisión (salida = 0.5) está formada por dos rectas: exactamente la geometría que la Sección 6.3 demostró que una unidad individual no puede producir.</p>

Los pesos impresos muestran cómo resolvió la red el problema: cada unidad oculta implementa una subfunción linealmente separable y la unidad de salida las combina. Cuál descomposición encuentra el descenso de gradiente depende de los pesos iniciales, un primer ejemplo de la no convexidad de la Sección 4.2. Minsky y Papert sabían en 1969 que esta arquitectura <em>representa</em> XOR; lo que dudaban era que algún procedimiento pudiera <em>aprender</em> los pesos de la capa oculta solo a partir de pares de entrada y salida, ya que nadie les dice a las unidades ocultas qué deben calcular.<sup class="c"><a href="#r3">[3]</a></sup> La retropropagación responde a esa duda: las unidades ocultas reciben su señal de error desde la salida mediante la regla de la cadena.

<div class="box key"><h4>Ejercicio 3</h4><p>Cambia la semilla en <code>set.seed(0)</code> a 1, 2, 3, 4 y vuelve a ejecutar. ¿Converge siempre la red en 10,000 épocas? Compara las pérdidas finales y las funciones aprendidas. Luego reemplaza el gradiente del error cuadrático <code>(A2 - y) * A2 * (1 - A2)</code> por el gradiente de la entropía cruzada <code>(A2 - y)</code> y observa cómo cambia la meseta del comienzo del entrenamiento. La Sección 8.2 explica por qué.</p></div>

### 8.6 Un MLP general en R base

El código de XOR deja fijas dos capas. Las funciones siguientes implementan las ecuaciones matriciales de las Secciones 8.1 y 8.3 para cualquier vector de tamaños de capa, con la opción de elegir la activación oculta (sigmoide o ReLU), inicialización de Glorot o gaussiana (Sección 9.3), una penalización L2 opcional (Sección 9.5) y una salida sigmoide entrenada con la entropía cruzada binaria. `nnet` hace lo mismo con un mejor optimizador (BFGS) y se usa a partir de la Sección 9.7; estas funciones son las que conviene leer línea por línea junto a las ecuaciones. Separan el <em>modelo</em> —cómo calcular la salida (`mlp_forward`) y el gradiente (`mlp_backward`)— del <em>bucle de entrenamiento</em> `mlp_train`, que implementa el <strong>descenso de gradiente estocástico por minilotes</strong>: en cada época baraja los casos de entrenamiento, los divide en lotes, da un paso de gradiente por lote y registra la pérdida de entrenamiento (y la de validación, si se entrega un conjunto de validación) después de cada época: las <em>curvas de pérdida</em> que son el principal instrumento de diagnóstico de la Sección 9.



In [ ]:
# Un perceptrón multicapa general para clasificación binaria: inicialización, hacia adelante, hacia atrás (R base, matrices al estilo numpy).
mlp_init <- function(sizes, act = "relu", init = "glorot", l2 = 0, seed = 0) {
  set.seed(seed); W <- list(); b <- list()
  for (l in seq_len(length(sizes) - 1)) {
    n_in <- sizes[l]; n_out <- sizes[l + 1]
    if (identical(init, "glorot")) { lim <- sqrt(6 / (n_in + n_out)); W[[l]] <- matrix(runif(n_in * n_out, -lim, lim), n_in, n_out) }
    else W[[l]] <- matrix(rnorm(n_in * n_out, 0, init), n_in, n_out)
    b[[l]] <- rep(0, n_out)
  }
  list(W = W, b = b, act = act, l2 = l2)
}
phi  <- function(net, z) if (net$act == "relu") pmax(z, 0) else sigmoid(z)
dphi <- function(net, z, a) if (net$act == "relu") (z > 0) * 1 else a * (1 - a)
mlp_forward <- function(net, X) {                             # devuelve Z, A (lista) y el vector de salida
  Z <- list(); A <- list(X)
  for (l in seq_along(net$W)) {
    z <- A[[l]] %*% net$W[[l]] + matrix(net$b[[l]], nrow(X), length(net$b[[l]]), byrow = TRUE)
    Z[[l]] <- z; A[[l + 1]] <- if (l == length(net$W)) sigmoid(z) else phi(net, z)
  }
  list(Z = Z, A = A, out = as.vector(A[[length(A)]]))
}
mlp_loss <- function(net, X, y) {
  p <- pmin(pmax(mlp_forward(net, X)$out, 1e-12), 1 - 1e-12)
  pen <- if (net$l2) 0.5 * net$l2 * sum(sapply(net$W, function(W) sum(W^2))) else 0
  -mean(y * log(p) + (1 - y) * log(1 - p)) + pen
}
mlp_backward <- function(net, X, y) {                         # devuelve dL/dW y dL/db para cada capa
  n <- length(y); fp <- mlp_forward(net, X); A <- fp$A; Z <- fp$Z
  delta <- (A[[length(A)]] - y) / n                           # Delta(L) = (Yhat - Y)/n para sigmoide + entropía cruzada
  gW <- vector("list", length(net$W)); gb <- vector("list", length(net$W))
  for (l in length(net$W):1) {
    gW[[l]] <- t(A[[l]]) %*% delta + net$l2 * net$W[[l]]; gb[[l]] <- colSums(delta)
    if (l > 1) delta <- (delta %*% t(net$W[[l]])) * dphi(net, Z[[l - 1]], A[[l]])
  }
  list(gW = gW, gb = gb)
}

# --- El bucle de entrenamiento: descenso de gradiente estocástico por minilotes, con la pérdida registrada tras cada época. ---
mlp_train <- function(net, X, y, lr = 0.1, epochs = 100, batch = 32, Xval = NULL, yval = NULL, seed = 0) {
  set.seed(seed); hist <- list(train = numeric(0), val = numeric(0))
  for (ep in 1:epochs) {
    idx <- sample(length(y)); folds <- split(idx, ceiling(seq_along(idx) / batch))
    for (b in folds) {
      g <- mlp_backward(net, X[b, , drop = FALSE], y[b])
      for (l in seq_along(net$W)) { net$W[[l]] <- net$W[[l]] - lr * g$gW[[l]]; net$b[[l]] <- net$b[[l]] - lr * g$gb[[l]] }
    }
    hist$train <- c(hist$train, mlp_loss(net, X, y))
    if (!is.null(Xval)) hist$val <- c(hist$val, mlp_loss(net, Xval, yval))
    if (!is.finite(tail(hist$train, 1))) break
  }
  list(net = net, hist = hist)
}
# Cáncer de mama, las 9 características: 80% entrenamiento / 20% prueba, estandarizadas solo con estadísticos del conjunto de entrenamiento.
set.seed(0); feat9 <- as.matrix(bc[, 1:9]); n <- nrow(feat9); tr <- sample(n, round(0.8 * n))
mu <- colMeans(feat9[tr, ]); sdv <- apply(feat9[tr, ], 2, sd)
Ztr <- scale(feat9[tr, ], mu, sdv); Zte <- scale(feat9[-tr, ], mu, sdv); ytr <- y_mal[tr]; yte <- y_mal[-tr]
cat(sprintf("training cases %d (malignant %d), test cases %d (malignant %d)\n", length(ytr), sum(ytr), length(yte), sum(yte)))
plot(NA, xlim = c(1, 100), ylim = c(0.03, 0.8), log = "y", xlab = "epoch", ylab = "cross-entropy loss", main = "Loss curves of the base-R MLP (breast cancer)")
for (cfg in list(c("sigmoid", ORANGE), c("relu", BLUE))) {
  res <- mlp_train(mlp_init(c(9, 16, 1), act = cfg[1]), Ztr, ytr, lr = 0.1, epochs = 100, Xval = Zte, yval = yte)
  lines(res$hist$train, col = cfg[2], lwd = 1.4); lines(res$hist$val, col = cfg[2], lwd = 1.4, lty = 2)
  acc <- mean((mlp_forward(res$net, Zte)$out >= 0.5) == yte)
  cat(sprintf("9-16-1 %-7s: final train loss %.4f, test loss %.4f, test accuracy %.3f\n",
              cfg[1], tail(res$hist$train, 1), tail(res$hist$val, 1), acc))
}
legend("topright", c("sigmoid train", "relu train", "(dashed = test)"), col = c(ORANGE, BLUE, GREY), lwd = 1.4, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 11.</b> Entropía cruzada de entrenamiento (línea continua) y de prueba (línea discontinua) de la red 9-16-1 con unidades ocultas sigmoides y ReLU, durante 100 épocas de SGD por minilotes con tasa de aprendizaje 0.1. Ambas redes alcanzan una exactitud en prueba de alrededor de 0.95 (sigmoide) a 0.96 (ReLU), cercana al nivel de los mejores métodos de las Unidades 3–4 con estos datos (la Sección 10 los ajusta como corresponde). La red ReLU desciende más rápido en las primeras épocas, como predice su derivada que no se satura. La pérdida de prueba se aplana mientras la de entrenamiento sigue bajando: la red empieza a ajustar la muestra de entrenamiento mejor que los casos nuevos, el sobreajuste que la Sección 9.2 aborda con la detención temprana.</p>

### 8.7 Comprobación del gradiente

Es fácil que el código de retropropagación quede sutilmente mal —una matriz traspuesta, una derivada faltante, un índice de capa desplazado en uno— y un gradiente erróneo a menudo igual hace bajar la pérdida, lentamente, de modo que el error pasa inadvertido. La prueba estándar compara el gradiente analítico con la diferencia finita centrada \(\left[L(\theta + \epsilon) - L(\theta - \epsilon)\right] / 2\epsilon\) para una muestra de parámetros, y reporta el error relativo \(|g_{\text{bp}} - g_{\text{fd}}| / (|g_{\text{bp}}| + |g_{\text{fd}}|)\). Una implementación correcta da errores relativos por debajo de alrededor de \(10^{-7}\) en doble precisión; un error de programación da errores del orden de 1.



In [ ]:
# Comprobación del gradiente: retropropagación frente a diferencias finitas centradas, para pesos aleatorios de cada capa.
set.seed(1); net_chk <- mlp_init(c(9, 8, 4, 1), act = "sigmoid", seed = 1)
Xs <- Ztr[1:25, ]; ys <- ytr[1:25]; g <- mlp_backward(net_chk, Xs, ys); eps <- 1e-5; rows <- list()
for (l in seq_along(net_chk$W)) for (rep in 1:4) {
  i <- sample(nrow(net_chk$W[[l]]), 1); j <- sample(ncol(net_chk$W[[l]]), 1); orig <- net_chk$W[[l]][i, j]
  net_chk$W[[l]][i, j] <- orig + eps; Lp <- mlp_loss(net_chk, Xs, ys)
  net_chk$W[[l]][i, j] <- orig - eps; Lm <- mlp_loss(net_chk, Xs, ys); net_chk$W[[l]][i, j] <- orig
  gfd <- (Lp - Lm) / (2 * eps); gbp <- g$gW[[l]][i, j]
  rows[[length(rows) + 1]] <- data.frame(layer = l, weight = sprintf("W[%d,%d]", i, j),
      backprop = gbp, finite_diff = gfd, rel_error = abs(gbp - gfd) / (abs(gbp) + abs(gfd) + 1e-15))
}
chk <- do.call(rbind, rows); print(chk, row.names = FALSE, digits = 3)
cat(sprintf("\nlargest relative error: %.2e\n", max(chk$rel_error)))



Todos los errores relativos son pequeños —la mayoría del orden de \(10^{-8}\) o menores—, de modo que la implementación de las cuatro ecuaciones de la retropropagación es correcta para una red de tres capas con entrada de 9 dimensiones, incluidas las capas que el ejemplo de XOR nunca puso a prueba.

<div class="box key"><h4>Idea clave</h4><p>La retropropagación no es una regla de aprendizaje, sino un método para calcular derivadas: la regla de la cadena aplicada capa por capa desde la salida hacia atrás, reutilizando la magnitud \(\delta^{(l)} = \partial L / \partial z^{(l)}\) de modo que el costo del gradiente completo sea aproximadamente el doble del costo de una pasada hacia adelante. Luego, el descenso de gradiente usa esas derivadas. Todo framework moderno automatiza exactamente el cálculo de la Sección 8.6 —para cualquier arquitectura, registrando las operaciones de la pasada hacia adelante y derivándolas en orden inverso (diferenciación automática).</p></div>

## 9. El entrenamiento en la práctica

La retropropagación entrega el gradiente; todo lo demás en el entrenamiento de una red es un conjunto de decisiones que la teoría no toma por ti. Esta sección las recorre en el orden en que suelen causar problemas, con un experimento para cada una, usando las funciones en R base con los datos de cáncer de mama y luego `nnet`.

### 9.1 La tasa de aprendizaje

La tasa de aprendizaje \(\eta\) es el hiperparámetro más importante del descenso de gradiente simple. Si es demasiado pequeña, el entrenamiento tarda miles de épocas en avanzar; si es demasiado grande, cada paso sobrepasa el mínimo, la pérdida oscila e incluso puede divergir hacia infinito: el comportamiento que se vio para el Adaline en la Figura 9, ahora en un problema no convexo en el que ninguna fórmula entrega el umbral. La práctica habitual es probar valores separados por factores de 10, elegir el mayor que haga bajar la pérdida de forma suave y reducirlo durante el entrenamiento. Los métodos adaptativos como Adam, que mantienen una estimación móvil de la media y la varianza del gradiente y escalan el paso para cada parámetro,<sup class="c"><a href="#r28">[28]</a></sup> son la opción por defecto en los frameworks de aprendizaje profundo de la Unidad 8. El paquete `nnet` de R evita la elección usando un optimizador cuasi-Newton (BFGS) que fija su propia longitud de paso, y `neuralnet` usa retropropagación resiliente (rprop); el bucle programado desde cero que sigue usa una \(\eta\) fija para que el fenómeno sea visible.



In [ ]:
# Tres tasas de aprendizaje en la red ReLU 9-16-1: demasiado pequeña (avanza a paso de tortuga), adecuada, demasiado grande (diverge).
plot(NA, xlim = c(1, 60), ylim = c(0.02, 50), log = "y", xlab = "epoch", ylab = "training cross-entropy (log scale)",
     main = "Learning rate: crawling, converging, diverging")
for (cfg in list(c(0.001, 1), c(0.1, 2), c(10, 3))) {
  lr <- cfg[1]; res <- mlp_train(mlp_init(c(9, 16, 1), act = "relu"), Ztr, ytr, lr = lr, epochs = 60)
  fin <- tail(res$hist$train, 1)
  status <- if (!is.finite(fin)) "diverged (loss became inf/nan)" else sprintf("final loss %.4f", fin)
  cat(sprintf("lr = %-6g: %2d epochs run, loss after epoch 1 = %.3f, %s\n", lr, length(res$hist$train), res$hist$train[1], status))
  lines(seq_along(res$hist$train), res$hist$train, type = "o", pch = 19, cex = 0.5, col = c(GREY, BLUE, ORANGE)[cfg[2]], lwd = 1.3)
}
legend("topright", c("eta = 0.001", "eta = 0.1", "eta = 10"), col = c(GREY, BLUE, ORANGE), lwd = 1.3, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 12.</b> Pérdida de entrenamiento para tres tasas de aprendizaje. Con \(\eta = 0.001\) la pérdida apenas se ha movido desde su inicialización después de 60 épocas (avanza a paso de tortuga); con \(\eta = 0.1\) cae de forma suave hasta un valor pequeño; con \(\eta = 10\) el primer paso lanza la pérdida por encima de 1 —los pasos sobrepasan el mínimo— y, aunque la ejecución se recupera en parte, se mantiene errática y muy por encima de la curva de \(\eta = 0.1\). Una tasa aún mayor hace que los pesos se desborden y que la pérdida quede indefinida, momento en el que el bucle de entrenamiento detiene la ejecución (la protección <code>is.finite</code>). Este es el equivalente no convexo del umbral exacto del Adaline de la Figura 9.</p>

### 9.2 Épocas y detención temprana

Cuánto tiempo entrenar es la segunda decisión, y es mejor tomarla observando un <strong>conjunto de validación</strong> —una parte de los datos de entrenamiento que se mantiene fuera de los pasos de gradiente— que fijarla de antemano. Por lo general, la pérdida de validación disminuye al principio junto con la de entrenamiento, luego se aplana y empieza a subir mientras la de entrenamiento sigue bajando: desde ese punto, la red está ajustando los casos de entrenamiento particulares en lugar del patrón que comparten. La <strong>detención temprana</strong> (early stopping) es la regla "detenerse cuando la pérdida de validación no ha mejorado durante \(k\) épocas (la <em>paciencia</em>) y conservar los pesos de la mejor época". Es la forma de regularización más barata y más usada para redes neuronales. `nnet` no tiene una opción de detención temprana —su optimizador BFGS corre hasta `maxit` o hasta su tolerancia—, así que el experimento siguiente usa el bucle programado desde cero, que registra la pérdida de validación en cada época; exagera el fenómeno entrenando una red ancha (9-256-1) con solo 80 casos.



In [ ]:
# Sobreajuste y detención temprana: una red ancha entrenada con una muestra pequeña; la pérdida de validación vuelve a subir.
set.seed(0); small <- sample(length(ytr), 80)
res <- mlp_train(mlp_init(c(9, 256, 1), act = "relu"), Ztr[small, ], ytr[small], lr = 0.1, epochs = 1500, batch = 16, Xval = Zte, yval = yte)
val <- res$hist$val; best <- which.min(val)
cat(sprintf("%d parameters trained on 80 cases\n", n_parameters(c(9, 256, 1))))
cat(sprintf("validation loss: minimum %.4f at epoch %d, final %.4f | training loss at end %.5f\n",
            min(val), best, tail(val, 1), tail(res$hist$train, 1)))
plot(res$hist$train, type = "l", col = BLUE, lwd = 1.3, log = "xy", xlab = "epoch (log)", ylab = "cross-entropy (log)",
     main = "Overfitting: the two loss curves separate", ylim = range(c(res$hist$train, val)))
lines(val, col = ORANGE, lwd = 1.3); abline(v = best, col = GREY, lty = 2)
legend("bottomleft", c("training (80 cases)", "validation (held out)", sprintf("early-stop epoch %d", best)),
       col = c(BLUE, ORANGE, GREY), lwd = c(1.3, 1.3, 1), lty = c(1, 1, 2), bty = "n", cex = 0.8)



<p class="caption"><b>Figura 13.</b> Pérdida de entrenamiento y de validación de una red 9-256-1 entrenada con 80 casos durante 1,500 épocas. La pérdida de entrenamiento disminuye hacia cero: la red puede memorizar 80 casos con miles de parámetros sin dificultad. La pérdida de validación alcanza su mínimo temprano y luego sube: cada época adicional hace que la red sea <em>peor</em> con pacientes nuevos mientras la hace mejor con los de entrenamiento. La detención temprana habría conservado los pesos de la época marcada. Los ejes log–log muestran que la separación comienza temprano y continúa durante toda la ejecución.</p>

### 9.3 Inicialización de los pesos

El descenso de gradiente debe partir de algún lugar, y el punto de partida importa más para las redes que para los modelos convexos. Iniciar todos los pesos en cero es fatal: cada unidad oculta recibe entonces la misma entrada y el mismo gradiente, de modo que todas las unidades de una capa permanecen idénticas para siempre: el problema de la <em>simetría</em>. Los pesos aleatorios rompen la simetría, pero su <em>escala</em> también importa: si son demasiado grandes, las unidades sigmoides y tanh se saturan de inmediato y sus gradientes se desvanecen; si son demasiado pequeños, la señal se encoge capa tras capa. Glorot y Bengio analizaron la varianza hacia adelante y hacia atrás a través de las capas y propusieron pesos extraídos de una distribución uniforme en \(\pm\sqrt{6 / (n_{\text{in}} + n_{\text{out}})}\), lo que mantiene ambas aproximadamente constantes a lo largo de una pila de capas tanh;<sup class="c"><a href="#r25">[25]</a></sup> esta inicialización de <strong>Glorot</strong> (Xavier) es la que la función `mlp_init` usa por defecto. Para unidades ReLU, que descartan la mitad de sus entradas, la varianza debe duplicarse (inicialización de He). La Sección 9.6 muestra lo que esta elección provoca en los gradientes de una red profunda.

### 9.4 Estandarización de las entradas

Las redes se entrenan con pasos de gradiente de un único tamaño en todas las direcciones del espacio de parámetros, y una característica cuya escala es mil veces la de otra produce gradientes mil veces mayores para sus pesos, de modo que ningún paso único sirve para ambas. Por eso, estandarizar cada característica a media 0 y desviación estándar 1 no es opcional para las redes, como sí lo es para los árboles: es parte del modelo. Todas las características de cáncer de mama comparten la escala ordinal de 1–10, así que, para ilustrar el punto de forma vívida, el bloque de código multiplica una característica por 1000 antes del ajuste y compara la exactitud por validación cruzada con y sin estandarización. El escalador debe ajustarse solo con los pliegues de entrenamiento: ajustarlo con todos los datos filtraría los estadísticos de los casos de prueba hacia el entrenamiento.



In [ ]:
# Estandarización de las entradas: la misma nnet con características en bruto (una inflada x1000) y estandarizadas, CV de 5 pliegues.
set.seed(0)
Xraw <- as.matrix(bc[, 1:9]); Xraw[, 1] <- Xraw[, 1] * 1000           # exagera la diferencia de escala
folds <- sample(rep(1:5, length.out = nrow(Xraw)))
cv_nnet_acc <- function(X, standardise) {
  acc <- sapply(1:5, function(k) {
    trk <- folds != k; tek <- !trk; Xt <- X[trk, ]; Xv <- X[tek, ]
    if (standardise) { m <- colMeans(Xt); s <- apply(Xt, 2, sd); Xt <- scale(Xt, m, s); Xv <- scale(Xv, m, s) }
    fit <- nnet(Xt, y_mal[trk], size = 16, entropy = TRUE, maxit = 200, trace = FALSE, MaxNWts = 5000)
    mean((predict(fit, Xv) >= 0.5) == y_mal[tek])
  })
  acc
}
raw <- cv_nnet_acc(Xraw, FALSE); std <- cv_nnet_acc(Xraw, TRUE)
cat(sprintf("raw features (1 blown up x1000): CV accuracy %.3f +/- %.3f\n", mean(raw), sd(raw)))
cat(sprintf("standardised features          : CV accuracy %.3f +/- %.3f\n", mean(std), sd(std)))



Con las características en bruto, el optimizador tiene dificultades: los gradientes de los pesos asociados a la característica inflada empequeñecen a los demás, y la red o bien no logra aprender o bien predice la clase mayoritaria, de modo que su exactitud por validación cruzada es baja e inestable. La misma arquitectura y los mismos pliegues con características estandarizadas dan una exactitud cercana a 0.96. De aquí en adelante, todo se estandariza dentro de la validación cruzada, ajustando el escalamiento solo con los pliegues de entrenamiento.

### 9.5 Regularización: decaimiento de pesos y dropout

La detención temprana limita el <em>tiempo</em> que tiene la red para ajustar el ruido; la regularización limita su <em>capacidad</em> para hacerlo. La <strong>penalización L2</strong> (decaimiento de pesos, weight decay) agrega \(\frac{\alpha}{2}\sum \theta^2\) a la pérdida, de modo que cada paso de gradiente también encoge cada peso hacia cero; se desalientan así los pesos grandes, que producen funciones abruptas y demasiado confiadas. Es el argumento `decay` de `nnet` y el argumento `l2` de las funciones en R base, y el mismo recurso que la penalización ridge de la Unidad 2. El <strong>dropout</strong>, propuesto por Srivastava y colaboradores, sigue otro camino: durante el entrenamiento, cada unidad oculta se pone a cero al azar con probabilidad \(p\) en cada pasada hacia adelante, de modo que la red no puede depender de ninguna unidad en particular y aprende características redundantes; en el momento de la prueba se usan todas las unidades, con las activaciones escaladas por \(1 - p\).<sup class="c"><a href="#r27">[27]</a></sup> El dropout puede leerse como el entrenamiento de un ensamble de redes adelgazadas que comparten pesos: la idea del bosque aleatorio de la Unidad 4 dentro de una sola red. Ni `nnet` ni `neuralnet` implementan dropout; los frameworks de la Unidad 8 sí lo hacen.

### 9.6 Gradientes que se desvanecen: por qué la profundidad era difícil

La Sección 8.4 observó que el gradiente que llegaba a la capa oculta era diez veces menor que el gradiente en la salida, porque se había multiplicado por un peso y por una derivada de la sigmoide \(\leq 0.25\). En una red con diez capas ocultas el factor se aplica diez veces, y el gradiente de la primera capa puede ser \(10^{-6}\) del gradiente de la última: las primeras capas, que deberían aprender las características más básicas, casi no reciben señal y apenas se mueven. Hochreiter identificó este <strong>problema del gradiente que se desvanece</strong> (vanishing gradient) en 1991,<sup class="c"><a href="#r29">[29]</a></sup> y es la razón principal por la que las redes con más de dos o tres capas ocultas eran poco prácticas antes de alrededor de 2010. Tres remedios rompieron la barrera: la activación ReLU, cuya derivada es exactamente 1 para las unidades activas;<sup class="c"><a href="#r26">[26]</a></sup> la inicialización escalada (Sección 9.3), que mantiene constante la varianza de la señal y del gradiente a través de las capas;<sup class="c"><a href="#r25">[25]</a></sup> y, para redes muy profundas, atajos arquitectónicos (conexiones residuales) que menciona la Unidad 8. El experimento mide la magnitud de los gradientes capa por capa en una red con diez capas ocultas en su inicialización.



In [ ]:
# Gradientes que se desvanecen: |dL/dW| medio por capa en la inicialización, para una red 9-[32 x 10]-1 con tres configuraciones.
sizes_deep <- c(9, rep(32, 10), 1)
configs <- list("sigmoid, Glorot" = list("sigmoid", "glorot"),
                "sigmoid, N(0,0.1)" = list("sigmoid", 0.1),
                "ReLU, Glorot" = list("relu", "glorot"))
rows <- list()
for (nm in names(configs)) {
  cf <- configs[[nm]]; net <- mlp_init(sizes_deep, act = cf[[1]], init = cf[[2]], seed = 0)
  g <- mlp_backward(net, Ztr[1:64, ], ytr[1:64]); rows[[nm]] <- sapply(g$gW, function(G) mean(abs(G)))
}
cat("ratio of first-layer to last-layer gradient magnitude:\n")
for (nm in names(configs)) cat(sprintf("  %-18s: %.1e\n", nm, rows[[nm]][1] / rows[[nm]][11]))
plot(NA, xlim = c(1, 11), ylim = range(unlist(rows)[unlist(rows) > 0]), log = "y", xlab = "layer (1 = first hidden, 11 = output)",
     ylab = "mean |dL/dW| (log scale)", main = "Gradient magnitude by layer (11-layer network)")
cols <- c(ORANGE, GREY, BLUE)
for (k in seq_along(configs)) lines(1:11, rows[[k]], type = "o", pch = 19, cex = 0.6, col = cols[k])
legend("bottomright", names(configs), col = cols, lwd = 1, pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 14.</b> Gradiente absoluto medio de los pesos de cada capa, en la inicialización, en una red con diez capas ocultas de 32 unidades. Con unidades sigmoides e inicialización de Glorot (naranja), el gradiente se encoge aproximadamente en un factor constante por capa y en la primera capa es varios órdenes de magnitud menor que en la salida; con pesos extraídos de \(\mathcal{N}(0, 0.1)\) (gris), el encogimiento es aún más rápido. Con unidades ReLU (azul), el gradiente se mantiene dentro de alrededor de un orden de magnitud en las once capas. Las primeras capas de la red sigmoide prácticamente no se entrenarían; las de la red ReLU sí. Las razones impresas sobre la figura hacen cuantitativa la comparación.</p>

<div class="box warn"><h4>Error frecuente</h4><p>"Más capas siempre es mejor". Una red más profunda tiene más capacidad, pero es más difícil de optimizar (gradientes que se desvanecen o que explotan), necesita más datos y, en problemas tabulares de unos pocos cientos de casos, rara vez mejora a una red con una capa oculta. La profundidad rinde cuando la entrada tiene estructura composicional —píxeles que forman bordes, bordes que forman partes, partes que forman objetos—, que es el tema de la Unidad 8. Para las características de cáncer de mama, una capa oculta es la opción por defecto adecuada, y la Sección 10 muestra que incluso esa apenas es necesaria.</p></div>

### 9.7 Hiperparámetros y selección de modelos con nnet

Las decisiones anteriores —arquitectura, activación, tasa de aprendizaje u optimizador, decaimiento de pesos, épocas— son <strong>hiperparámetros</strong>: magnitudes que se fijan antes del entrenamiento y que el propio entrenamiento no puede determinar. Se seleccionan con la metodología del Suplemento S1: validación cruzada sobre los datos de entrenamiento, nunca sobre el conjunto de prueba. Desde aquí, el notebook usa `nnet`, cuyos principales hiperparámetros son `size` (unidades ocultas) y `decay` (la intensidad de L2). La primera pregunta es el tamaño de la capa oculta, que se examina con una <em>curva de validación</em>: la exactitud por validación cruzada en función de un hiperparámetro.



In [ ]:
# Curva de validación: exactitud por CV de 5 pliegues de una nnet con una capa oculta en función del número de unidades ocultas.
set.seed(0); feat_bc <- as.matrix(bc[, 1:9]); folds_bc <- sample(rep(1:5, length.out = nrow(feat_bc)))
cv_size <- function(h) {
  sapply(1:5, function(k) {
    trk <- folds_bc != k; m <- colMeans(feat_bc[trk, ]); s <- apply(feat_bc[trk, ], 2, sd)
    Xt <- scale(feat_bc[trk, ], m, s); Xv <- scale(feat_bc[!trk, ], m, s)
    fit <- nnet(Xt, y_mal[trk], size = h, entropy = TRUE, decay = 1e-4, maxit = 300, trace = FALSE, MaxNWts = 5000)
    mean((predict(fit, Xv) >= 0.5) == y_mal[!trk])
  })
}
hsizes <- c(1, 2, 4, 8, 16, 32); means <- sapply(hsizes, function(h) mean(cv_size(h))); sds <- sapply(hsizes, function(h) sd(cv_size(h)))
curve_tab <- data.frame(hidden_units = hsizes, CV_accuracy = round(means, 3), sd = round(sds, 3),
                        parameters = sapply(hsizes, function(h) n_parameters(c(9, h, 1))))
print(curve_tab, row.names = FALSE)
plot(hsizes, means, type = "o", pch = 19, col = BLUE, log = "x", ylim = c(0.93, 1), xlab = "hidden units",
     ylab = "5-fold CV accuracy", main = "Validation curve over the hidden-layer size")
arrows(hsizes, means - sds, hsizes, means + sds, angle = 90, code = 3, length = 0.03, col = BLUE)



<p class="caption"><b>Figura 15.</b> Exactitud por validación cruzada (con una desviación estándar entre pliegues) de la red 9-\(h\)-1 para \(h\) entre 1 y 32. Una sola unidad oculta —una regresión logística sobre una característica aprendida— ya obtiene alrededor de 0.97, y la curva es prácticamente plana: dentro de la desviación estándar entre pliegues, ningún tamaño es claramente mejor que otro. Con estos datos, el tamaño de la capa oculta no es un hiperparámetro sensible.</p>

### 9.8 Clasificación multiclase: los glifos simulados

Con \(K > 2\) clases, la capa de salida tiene \(K\) unidades softmax y la pérdida es la entropía cruzada categórica (Sección 8.2); `nnet` hace esto cuando recibe una matriz de objetivos one-hot y `softmax = TRUE`. R no tiene un conjunto de datos de dígitos manuscritos, así que el `load_digits`<sup class="c"><a href="#r31">[31]</a></sup> de la edición en Python se reemplaza por los glifos simulados de 8 × 8 de la Sección 4.3: un problema de cinco clases con una entrada de 64 dimensiones, el escenario de la Unidad 8 en miniatura.



In [ ]:
# Una red softmax 64-32-5 con los glifos simulados: exactitud por validación cruzada y luego una mirada a los errores de prueba.
set.seed(0); fdig <- sample(rep(1:5, length.out = nrow(Zd)))
cv_d <- sapply(1:5, function(k) {
  fit <- nnet(Zd[fdig != k, ], class.ind(dig$y[fdig != k]), size = 32, softmax = TRUE, maxit = 200, trace = FALSE, MaxNWts = 5000)
  mean((max.col(predict(fit, Zd[fdig == k, ])) - 1) == (as.integer(dig$y[fdig == k]) - 1))
})
set.seed(0); fit_d <- nnet(Zd_tr, Ytr, size = 32, softmax = TRUE, maxit = 300, trace = FALSE, MaxNWts = 5000)
pred_d <- max.col(predict(fit_d, Zd_te)) - 1; truth_d <- as.integer(yd_te) - 1; wrong <- which(pred_d != truth_d)
cat(sprintf("64-32-5 network: %d parameters | CV accuracy %.3f +/- %.3f | test accuracy %.3f (%d errors in %d images)\n",
            n_parameters(c(64, 32, 5)), mean(cv_d), sd(cv_d), mean(pred_d == truth_d), length(wrong), length(truth_d)))
labs <- names(templates)
par(mfrow = c(2, 8), mar = c(1.6, 0.5, 1.6, 0.5))
for (i in head(wrong, 16)) {
  image(matrix(dig$X[-tr_d, ][i, ], 8, 8)[, 8:1], col = grey.colors(10, 1, 0), axes = FALSE)
  title(sprintf("t:%s p:%s", labs[truth_d[i] + 1], labs[pred_d[i] + 1]), cex.main = 0.7)
}
if (length(wrong) < 16) for (i in seq_len(16 - length(wrong))) plot.new()



<p class="caption"><b>Figura 16.</b> Los glifos de prueba que la red 64-32-5 clasificó mal, con la clase verdadera (t) y la predicha (p). La mayoría están muy corrompidos por el ruido simulado: las plantillas que comparten píxeles (la caja rellena y la mancha, las dos barras) son las que se confunden. La exactitud de la red, muy por encima de 0.9 en imágenes reservadas, está en línea con los resultados de k-NN y SVM de la Unidad 3 en datos comparables; con píxeles en bruto como entrada, ningún método de las unidades anteriores tiene una ventaja decisiva, y son las redes convolucionales de la Unidad 8 las que reducen aún más el error.</p>

### 9.9 Regresión: una red frente a la regresión lineal

La misma red con una unidad de salida lineal y la pérdida de error cuadrático es un modelo de regresión (`nnet` con `linout = TRUE`). La edición en Python usó los datos de diabetes de sklearn,<sup class="c"><a href="#r30">[30]</a></sup> que R no incluye; el sustituto en R son los datos de <strong>viviendas de Boston</strong> (Boston housing) de `mlbench`<sup class="c"><a href="#r38">[38]</a></sup>: 506 sectores censales, 13 variables y el valor mediano de las viviendas como variable a predecir. La pregunta de la edición en Python se mantiene: ¿lo hace mejor un modelo no lineal que la regresión lineal?



In [ ]:
# nnet (salida lineal) frente a regresión lineal en Boston housing: R^2 por validación cruzada de 5 pliegues.
data(BostonHousing, package = "mlbench"); B <- BostonHousing; B$chas <- as.numeric(as.character(B$chas))
Xb13 <- as.matrix(B[, 1:13]); yb <- B$medv; set.seed(0); fb <- sample(rep(1:5, length.out = nrow(Xb13)))
R2 <- function(pred, obs) 1 - sum((obs - pred)^2) / sum((obs - mean(obs))^2)
run_cv <- function(fit_pred) sapply(1:5, function(k) {
  trk <- fb != k; m <- colMeans(Xb13[trk, ]); s <- apply(Xb13[trk, ], 2, sd)
  Xt <- scale(Xb13[trk, ], m, s); Xv <- scale(Xb13[!trk, ], m, s); R2(fit_pred(Xt, yb[trk], Xv), yb[!trk])
})
res_lm   <- run_cv(function(Xt, yt, Xv) { d <- data.frame(Xt, y = yt); predict(lm(y ~ ., d), data.frame(Xv)) })
res_n8   <- run_cv(function(Xt, yt, Xv) predict(nnet(Xt, yt, size = 8,  linout = TRUE, decay = 0.1, maxit = 500, trace = FALSE, MaxNWts = 3000), Xv))
res_n32  <- run_cv(function(Xt, yt, Xv) predict(nnet(Xt, yt, size = 32, linout = TRUE, decay = 0.1, maxit = 500, trace = FALSE, MaxNWts = 3000), Xv))
print(data.frame(model = c("linear regression (U2)", "nnet 13-8-1, decay 0.1", "nnet 13-32-1, decay 0.1"),
                 CV_R2 = round(c(mean(res_lm), mean(res_n8), mean(res_n32)), 3),
                 sd = round(c(sd(res_lm), sd(res_n8), sd(res_n32)), 3)), row.names = FALSE)



Aquí la respuesta honesta difiere de la del ejemplo de diabetes de la edición en Python, y la diferencia es instructiva. Con los datos de Boston, una red no lineal modesta <em>sí</em> mejora a la regresión lineal —el \(R^2\) por validación cruzada sube de alrededor de 0.70 a aproximadamente 0.8—, porque las relaciones entre los predictores y el valor de las viviendas son genuinamente no lineales y 506 sectores bastan para revelarlo. Con los datos de diabetes de sklearn, con 442 pacientes y una señal casi lineal, la red no aportaba nada. Mismo procedimiento, conclusión opuesta: la lección metodológica es idéntica y es lo esencial: hacer la comparación con la línea base lineal, informar lo que muestran los datos y no dar nada por supuesto en ninguna de las dos direcciones.

## 10. Ejemplo resuelto: un MLP para el diagnóstico de cáncer de mama, de principio a fin

La subunidad 6.8 pide un ejemplo resuelto completo. Esta sección construye, selecciona, evalúa e interpreta una red neuronal para el problema diagnóstico recurrente —maligno frente a benigno a partir de las nueve características nucleares<sup class="c"><a href="#r13">[13]</a></sup>— siguiendo en cada paso el protocolo del Suplemento S1, y termina comparándola de forma justa con los métodos de las Unidades 3–4. Las decisiones de diseño, tomadas antes de mirar cualquier resultado:

1. **División de los datos.** La división 80/20 de la Sección 8.6 (`tr`: 546 casos de entrenamiento y 137 de prueba). Los casos de prueba se usan una sola vez, al final.
2. **Preprocesamiento.** Estandarización ajustada solo con los pliegues de entrenamiento.
3. **Arquitectura y pérdida.** Una capa oculta de unidades logísticas, una salida sigmoide, entropía cruzada binaria: `nnet` con `entropy = TRUE`.
4. **Regularización.** Decaimiento de pesos (`decay`), el equivalente en `nnet` de la penalización L2; `nnet` no tiene opción de detención temprana, así que el decaimiento y la propia convergencia del optimizador se encargan de la regularización.
5. **Selección del modelo.** Una grilla sobre el tamaño de la capa oculta y el decaimiento, puntuada con el ROC-AUC por validación cruzada de 5 pliegues en el conjunto de entrenamiento; AUC en lugar de exactitud porque el umbral de operación es una decisión clínica que se toma después.
6. **Evaluación.** En el conjunto de prueba: matriz de confusión, sensibilidad, especificidad, AUC y calibración.
7. **Interpretación.** Importancia por permutación: en qué características se apoya la red.
8. **Comparación.** La misma validación cruzada aplicada a la regresión logística, Bayes ingenuo, SVM, k-NN, un árbol de decisión y un bosque aleatorio.



In [ ]:
# Paso 5 - selección del modelo por AUC con validación cruzada de 5 pliegues sobre tamaños ocultos y decaimiento.
set.seed(0); fg <- sample(rep(1:5, length.out = length(ytr)))
cv_auc <- function(size, decay) {
  mean(sapply(1:5, function(k) {
    trk <- fg != k; fit <- nnet(Ztr[trk, ], ytr[trk], size = size, entropy = TRUE, decay = decay,
                                maxit = 300, trace = FALSE, MaxNWts = 5000)
    as.numeric(auc(roc(ytr[!trk], as.vector(predict(fit, Ztr[!trk, ])), quiet = TRUE)))
  }))
}
grid <- expand.grid(size = c(4, 8, 16), decay = c(1e-4, 1e-2, 1))
grid$CV_AUC <- round(mapply(cv_auc, grid$size, grid$decay), 4)
print(reshape(grid, idvar = "size", timevar = "decay", direction = "wide"), row.names = FALSE)
best <- grid[which.max(grid$CV_AUC), ]
cat(sprintf("\nbest configuration: size = %d, decay = %g -> CV AUC %.4f\n", best$size, best$decay, best$CV_AUC))
set.seed(0); mlp_final <- nnet(Ztr, ytr, size = best$size, entropy = TRUE, decay = best$decay, maxit = 500, trace = FALSE, MaxNWts = 5000)



La tabla vuelve a mostrar el paisaje plano de la Figura 15: todas las configuraciones alcanzan un AUC por validación cruzada de alrededor de 0.98–0.996, y las diferencias entre ellas son del orden de la desviación estándar entre pliegues. Un decaimiento más fuerte (1) es de hecho marginalmente el mejor con estos datos fácilmente separables, pero el margen está dentro del ruido. `cv_auc` elige la mejor media; ese modelo se reajusta con todos los casos de entrenamiento, y ese único modelo pasa al conjunto de prueba. A diferencia de la edición en Python, la grilla no tiene una dimensión de detención temprana, porque `nnet` no ofrece detención temprana: es una diferencia de biblioteca, no un cambio de método; aquí el decaimiento de pesos cumple el papel regularizador.



In [ ]:
# Paso 6 - evaluación en los casos de prueba reservados: matriz de confusión, sensibilidad, especificidad, AUC, calibración.
p_te <- as.vector(predict(mlp_final, Zte)); pred_te <- as.integer(p_te >= 0.5)
cm <- table(truth = yte, predicted = pred_te)
tn <- cm["0", "0"]; fp <- cm["0", "1"]; fn <- cm["1", "0"]; tp <- cm["1", "1"]
roc_mlp <- roc(yte, p_te, quiet = TRUE); auc_mlp <- as.numeric(auc(roc_mlp))
cat(sprintf("test set: TP %d, FN %d, TN %d, FP %d | accuracy %.3f, sensitivity %.3f, specificity %.3f, AUC %.4f\n",
            tp, fn, tn, fp, (tp + tn) / length(yte), tp / (tp + fn), tn / (tn + fp), auc_mlp))
co <- coords(roc_mlp, "all", ret = c("threshold", "sensitivity", "specificity"), transpose = FALSE)
i95 <- which(co$sensitivity >= 0.95); i95 <- i95[which.max(co$specificity[i95])]
cat(sprintf("threshold for sensitivity >= 0.95: %.3f -> sensitivity %.3f, specificity %.3f\n",
            co$threshold[i95], co$sensitivity[i95], co$specificity[i95]))
bins <- cut(p_te, quantile(p_te, seq(0, 1, 0.2)), include.lowest = TRUE)
cal <- data.frame(mean_pred = tapply(p_te, bins, mean), obs_frac = tapply(yte, bins, mean))
par(mfrow = c(1, 3), mar = c(4, 4, 2, 1))
mosaicplot(cm, color = c(BLUE, ORANGE), main = "Confusion matrix (threshold 0.5)")
plot(1 - roc_mlp$specificities, roc_mlp$sensitivities, type = "l", col = BLUE, lwd = 1.6, xlab = "1 - specificity",
     ylab = "sensitivity", main = sprintf("ROC curve (AUC %.3f)", auc_mlp)); abline(0, 1, col = GREY, lty = 2)
points(1 - co$specificity[i95], co$sensitivity[i95], pch = 19, col = ORANGE)
plot(cal$mean_pred, cal$obs_frac, type = "o", pch = 19, col = BLUE, xlim = c(0, 1), ylim = c(0, 1),
     xlab = "mean predicted P(malignant)", ylab = "observed fraction", main = "Calibration"); abline(0, 1, col = GREY, lty = 2)



<p class="caption"><b>Figura 17.</b> Evaluación de la red seleccionada en los casos reservados. Izquierda: la matriz de confusión con el umbral por defecto de 0.5, con unos pocos errores en cada dirección. Centro: la curva ROC, con un AUC cercano a 0.99; el punto naranja es el umbral más bajo con el que se detecta al menos el 95 % de los casos malignos, el punto de operación que elegiría una aplicación de tamizaje, un recordatorio de que 0.5 no es un umbral privilegiado. Derecha: la curva de calibración, la fracción observada de casos malignos frente a la probabilidad predicha media en cinco intervalos de cuantiles; puntos cercanos a la diagonal significan que las probabilidades de la red pueden leerse como probabilidades. Las redes entrenadas con entropía cruzada suelen estar razonablemente bien calibradas, a diferencia del Bayes ingenuo (Unidad 3), cuyas probabilidades a posteriori se empujan hacia los extremos;<sup class="c"><a href="#r32">[32]</a></sup> sin embargo, los pocos casos de prueba por intervalo hacen que la curva sea ruidosa, y una afirmación sobre la calibración necesitaría más datos.</p>

Leyendo los números como lo haría un ingeniero: el conjunto de prueba contiene unas pocas decenas de casos malignos, de modo que cada falso negativo vale más de dos puntos porcentuales de sensibilidad, y el intervalo de confianza de cualquiera de estas estimaciones es amplio (Suplemento S1). El resumen honesto es "AUC de alrededor de 0.99, sensibilidad de alrededor de 0.90 y especificidad de alrededor de 0.98 con el umbral por defecto —y un umbral más bajo alcanza un 100 % de sensibilidad con un pequeño costo en especificidad—, en unos 137 casos de una sola fuente", seguido de la pregunta de la Unidad 1: ¿se mantendría con punciones de otro laboratorio?



In [ ]:
# Paso 7 - interpretación por importancia por permutación: ¿cuánto cae el AUC de prueba cuando se baraja una característica?
set.seed(0); base_auc <- auc_mlp
imp <- sapply(1:9, function(j) {
  drops <- replicate(30, {
    Zp <- Zte; Zp[, j] <- sample(Zp[, j])
    base_auc - as.numeric(auc(roc(yte, as.vector(predict(mlp_final, Zp)), quiet = TRUE)))
  })
  c(mean = mean(drops), sd = sd(drops))
})
imp <- data.frame(feature = colnames(feat9), mean_AUC_drop = imp["mean", ], sd = imp["sd", ])
imp <- imp[order(-imp$mean_AUC_drop), ]
print(head(round(imp[, 2:3], 4)[order(-imp$mean_AUC_drop)[1:6], ], 4))
print(imp[1:6, ], row.names = FALSE, digits = 3)
top <- imp[6:1, ]
par(mar = c(4, 9, 2, 1)); barplot(top$mean_AUC_drop, names.arg = top$feature, horiz = TRUE, las = 1, col = BLUE,
        xlab = "decrease in test AUC when the feature is permuted", main = "Permutation importance of the selected MLP")



<p class="caption"><b>Figura 18.</b> Las características cuya permutación aleatoria más reduce el AUC de prueba de la red. Un pequeño número de mediciones de la forma nuclear concentra casi todo el poder discriminativo de la red, en concordancia con la patología; las características restantes tienen una importancia cercana a cero, ya sea porque la red no las necesita o porque su información está duplicada en características correlacionadas (el tamaño celular, la forma celular y el grosor de los cúmulos varían juntos), en cuyo caso permutar una deja que las otras transporten la señal. La importancia por permutación mide dependencia, no causalidad, y reparte la señal entre las características correlacionadas de forma arbitraria.</p>

El paso final es la comparación que todo artículo aplicado debería contener y que muchos omiten: el mismo protocolo aplicado a los métodos más simples de las Unidades 2–4.



In [ ]:
# Paso 8 - la comparación justa: AUC por CV de 5 pliegues en el conjunto de entrenamiento y luego AUC de prueba, para siete clasificadores.
suppressPackageStartupMessages({ library(e1071); library(rpart); library(class); library(randomForest) })
score_test <- function(name, trainf) {
  cv <- sapply(1:5, function(k) trainf(Ztr[fg != k, ], ytr[fg != k], Ztr[fg == k, ], ytr[fg == k]))
  te <- trainf(Ztr, ytr, Zte, yte)
  c(CV_AUC = mean(cv), sd = sd(cv), test_AUC = te)
}
auc_of <- function(y, s) as.numeric(auc(roc(y, s, quiet = TRUE)))
models <- list(
  "logistic regression (U2)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(glm(yt ~ ., data.frame(Xt, yt = yt), family = binomial), data.frame(Xv), type = "response")),
  "Gaussian naive Bayes (U3)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(naiveBayes(Xt, factor(yt)), Xv, type = "raw")[, 2]),
  "SVM, RBF kernel (U3)" = function(Xt, yt, Xv, yv) auc_of(yv, attr(predict(svm(Xt, factor(yt), probability = TRUE), Xv, probability = TRUE), "probabilities")[, "1"]),
  "k-NN, k = 7 (U3)" = function(Xt, yt, Xv, yv) { pr <- knn(Xt, Xv, factor(yt), k = 7, prob = TRUE); pw <- attr(pr, "prob"); auc_of(yv, ifelse(pr == "1", pw, 1 - pw)) },
  "decision tree (U4)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(rpart(yt ~ ., data.frame(Xt, yt = factor(yt)), method = "class", control = rpart.control(maxdepth = 4)), data.frame(Xv), type = "prob")[, 2]),
  "random forest, 200 trees (U4)" = function(Xt, yt, Xv, yv) auc_of(yv, predict(randomForest(Xt, factor(yt), ntree = 200), Xv, type = "prob")[, 2]),
  "MLP, selected above (U6)" = function(Xt, yt, Xv, yv) auc_of(yv, as.vector(predict(nnet(Xt, yt, size = best$size, entropy = TRUE, decay = best$decay, maxit = 500, trace = FALSE, MaxNWts = 5000), Xv))))
set.seed(0); comp <- t(sapply(names(models), function(nm) score_test(nm, models[[nm]])))
print(round(as.data.frame(comp), 3))



La tabla es el resultado honesto del ejemplo resuelto. La regresión logística, la SVM, el bosque aleatorio y la red alcanzan todos un AUC por validación cruzada cercano o superior a 0.99, y las diferencias entre ellos están dentro de una desviación estándar entre pliegues; el árbol de decisión individual es el más débil, y el Bayes ingenuo es competitivo en AUC, aunque sus probabilidades están mal calibradas, como mostró la Unidad 3. Con unos pocos cientos de casos y nueve características diseñadas, la flexibilidad adicional de la red neuronal no aporta nada que una regresión logística regularizada no tenga ya; y la regresión logística tiene diez coeficientes interpretables, se entrena en milisegundos y no tiene hiperparámetros que valga la pena ajustar. La red se gana su lugar cuando la entrada es la imagen de la punción y no nueve números calculados a partir de ella: el tema de la Unidad 8.

<div class="box key"><h4>Qué enseña el ejemplo resuelto</h4>
<ul>
<li>El protocolo (división → estandarización dentro de la CV → selección por validación cruzada → una única evaluación en prueba → interpretación → comparación) es el mismo para todos los modelos del curso; la red agrega hiperparámetros, no pasos.</li>
<li>Reporta el AUC con su variabilidad entre pliegues, el punto de operación relevante para el uso clínico y la calibración; una única exactitud no es una evaluación.</li>
<li>Incluye siempre las líneas base simples. Un resultado que no supera a la regresión logística por más que el ruido es un resultado a favor de la regresión logística.</li>
</ul></div>

<div class="box key"><h4>Ejercicio 4 (preparación para el Laboratorio 4)</h4><p>Repite el ejemplo resuelto con los glifos simulados (cinco clases): selecciona el tamaño de la capa oculta y <code>decay</code> por exactitud con validación cruzada, evalúa en la partición reservada de la Sección 4.3 con una matriz de confusión de 5 × 5 y compara con k-NN y un bosque aleatorio. Luego reduce el conjunto de entrenamiento al 10 % de su tamaño y repite: ¿qué método se degrada menos, y por qué?</p></div>

## 11. Del perceptrón multicapa al aprendizaje profundo

### 11.1 Profundidad

Una red <strong>profunda</strong> es un perceptrón multicapa con muchas capas ocultas; no hay ninguna otra diferencia de principio. El teorema de aproximación universal dice que basta con una capa oculta para <em>representar</em> cualquier función, pero puede necesitar un número exponencial de unidades para hacerlo, mientras que una red profunda puede representar la misma función con muchos menos parámetros <em>componiendo</em> funciones simples: las unidades de cada capa reutilizan las características de la capa inferior, como un programa reutiliza subrutinas. La revisión de LeCun, Bengio y Hinton enuncia la tesis del campo: las redes profundas aprenden una jerarquía de representaciones, cada nivel más abstracto que el anterior, y esta jerarquía es lo que las características diseñadas a mano del reconocimiento de patrones clásico nunca pudieron proporcionar.<sup class="c"><a href="#r6">[6]</a></sup> Lo que hizo práctica la profundidad tras décadas de fracasos fue la conjunción de los remedios de la Sección 9 —ReLU, inicialización escalada, dropout, Adam— con dos factores externos: conjuntos de datos etiquetados de millones de ejemplos y las unidades de procesamiento gráfico (GPU), cuyos miles de núcleos simples ejecutan en paralelo los productos de matrices de la propagación hacia adelante y hacia atrás, cien veces más rápido que un procesador de propósito general. La red de ImageNet de 2012 se entrenó en dos GPU durante alrededor de una semana.<sup class="c"><a href="#r5">[5]</a></sup>

### 11.2 Convolución: un adelanto de la Unidad 8

Una primera capa totalmente conectada sobre una imagen de 256 × 256 tiene 65,536 entradas por unidad, y una capa oculta de 1,000 unidades tiene 65 millones de pesos: demasiados para aprenderlos de cualquier conjunto de datos médicos, y un desperdicio, porque la característica que detecta el margen de un tumor en la esquina superior izquierda es la misma que lo detecta en la inferior derecha. Una <strong>capa convolucional</strong> resuelve ambos problemas: cada unidad mira solo una pequeña ventana de la imagen (por ejemplo, 3 × 3 píxeles), y todas las unidades de la capa comparten los mismos pesos, deslizando la ventana a lo largo de la imagen. La capa tiene entonces 9 pesos en lugar de 65 millones, detecta su característica dondequiera que aparezca, y las pilas de tales capas aprenden la jerarquía bordes → texturas → partes → objetos. LeCun introdujo la arquitectura para el reconocimiento de dígitos manuscritos a fines de la década de 1980 y le dio su forma definitiva, LeNet-5, en 1998;<sup class="c"><a href="#r33">[33]</a></sup> cada sistema de imágenes médicas de la Sección 5 es un descendiente de ella. El bloque de código siguiente aplica dos filtros de 3 × 3 diseñados a mano a un glifo para mostrar lo que calcula una unidad convolucional; en la Unidad 8 los filtros se aprenden por retropropagación exactamente como se aprendieron los pesos de esta unidad.



In [ ]:
# Lo que calcula una unidad convolucional: un filtro de 3x3 deslizado sobre una imagen (dos detectores de bordes de Sobel diseñados a mano).
conv2 <- function(img, k) {
  kk <- (nrow(k) - 1) / 2; pad <- matrix(0, nrow(img) + 2 * kk, ncol(img) + 2 * kk)
  pad[(kk + 1):(kk + nrow(img)), (kk + 1):(kk + ncol(img))] <- img; out <- matrix(0, nrow(img), ncol(img))
  for (i in seq_len(nrow(img))) for (j in seq_len(ncol(img))) out[i, j] <- sum(pad[i:(i + 2 * kk), j:(j + 2 * kk)] * k)
  out
}
img <- templates$box                                                           # un glifo de 8 x 8
filters <- list("vertical edges" = matrix(c(-1, 0, 1, -2, 0, 2, -1, 0, 1), 3, byrow = TRUE),
                "horizontal edges" = matrix(c(-1, -2, -1, 0, 0, 0, 1, 2, 1), 3, byrow = TRUE))
cat(sprintf("a 3x3 convolutional unit has %d parameters whatever the image size; a fully connected unit on this 8x8 image has %d, on a 256x256 image %d\n",
            3 * 3 + 1, 8 * 8 + 1, 256 * 256 + 1))
par(mfrow = c(1, 3), mar = c(1, 1, 2, 1))
image(img[, 8:1], col = grey.colors(10, 1, 0), axes = FALSE, main = "input glyph (8x8)")
for (nm in names(filters)) { r <- conv2(img, filters[[nm]]); image(r[, 8:1], col = hcl.colors(20, "Blue-Red"), axes = FALSE, main = paste("response to", nm)) }



<p class="caption"><b>Figura 19.</b> Un glifo simulado (el contorno de una caja) y sus respuestas a dos filtros de 3 × 3 (operadores de Sobel), cada uno aplicado en todas las posiciones de la imagen. El mapa de respuesta de la izquierda se ilumina donde la intensidad cambia de izquierda a derecha —los lados verticales de la caja— y el de la derecha donde cambia de arriba hacia abajo —los lados horizontales—. Cada mapa es la salida de una unidad convolucional, con nueve pesos, evaluada en 64 posiciones. Una red convolucional aprende a partir de los datos docenas de filtros como estos en su primera capa y luego combina sus respuestas en la siguiente.</p>

### 11.3 Secuencias y atención

Otras dos arquitecturas completan el mapa. Las <strong>redes recurrentes</strong> (Sección 3.1) procesan una secuencia elemento por elemento, arrastrando hacia adelante un estado oculto, y fueron el modelo estándar para señales y lenguaje durante la década de 2010; la unidad de <em>memoria de corto y largo plazo</em> (long short-term memory) de Hochreiter y Schmidhuber resolvió su problema del gradiente que se desvanece a través del tiempo mediante celdas de memoria con compuertas,<sup class="c"><a href="#r34">[34]</a></sup> y las redes recurrentes leen electrocardiogramas, electroencefalogramas y series de tiempo clínicas. El <strong>Transformer</strong> de 2017 reemplazó la recurrencia por la <em>atención</em>, un mecanismo en el que cada elemento de la secuencia calcula una suma ponderada sobre todos los demás, con pesos que son a su vez funciones aprendidas del contenido; como el cálculo sobre las posiciones es paralelo y no secuencial, los Transformers se entrenan de forma eficiente con secuencias muy largas y corpus muy grandes,<sup class="c"><a href="#r35">[35]</a></sup> y son la base de los grandes modelos de lenguaje y de los predictores de estructura de proteínas de la Unidad 1. Ambas son, al nivel de esta unidad, disposiciones de sumas ponderadas y funciones de activación entrenadas por retropropagación; lo que cambia es el patrón de conexiones y de pesos compartidos.

<div class="box bridge"><h4>Puente con la Unidad 8</h4><p>La Unidad 8 entrena una red convolucional para la clasificación de patrones en imágenes médicas. Todo lo que necesita está en esta unidad: la capa como producto de matrices, la salida softmax y la pérdida de entropía cruzada, la retropropagación, la tasa de aprendizaje y la detención temprana, la estandarización de la entrada y el protocolo de evaluación de la Sección 10. La capa convolucional de la Figura 19 es el único componente nuevo.</p></div>

## 12. Resumen

Ahora deberías ser capaz de explicar, sin apuntes:

- El cerebro en cifras —unos 86 mil millones de neuronas, \(10^{14}\)–\(10^{15}\) sinapsis, componentes que operan en milisegundos, paralelismo masivo— y las cinco propiedades que toman de él las redes artificiales (unidades simples, conexiones ponderadas, representación distribuida, degradación suave, aprendizaje mediante el cambio de pesos), junto con la afirmación honesta de que una red artificial está inspirada en el cerebro, pero no es un modelo de él.
- Los componentes de una unidad —entradas, pesos, sesgo, suma ponderada, función de activación, salida— y de una red —capas de entrada, ocultas y de salida, arquitectura, prealimentada frente a recurrente— y cómo contar los parámetros de una arquitectura.
- Por qué una neurona sigmoide individual es una regresión logística, y por qué una red con capas ocultas es una regresión logística sobre características aprendidas.
- El teorema de aproximación universal (lo que promete y lo que no) y las cuatro limitaciones: necesidad de muchos datos, opacidad, muchos hiperparámetros, entrenamiento no convexo.
- Las funciones de activación escalón, sigmoide, tanh, ReLU y softmax, sus derivadas, y por qué la derivada importa (saturación, unidades muertas, gradientes que se desvanecen).
- Cómo una unidad calcula AND y OR, la demostración geométrica y algebraica de que ninguna calcula XOR, y por qué eso motiva las capas ocultas.
- La regla de aprendizaje de Rosenblatt, su derivación como un paso a lo largo de \(y\mathbf{x}\), el teorema de convergencia y su cota \((R/\gamma)^2\), el seguimiento a mano sobre AND, el comportamiento con datos no separables y la regla delta del Adaline como descenso de gradiente sobre una pérdida suave.
- La propagación hacia adelante como productos de matrices; las pérdidas de entropía cruzada y de error cuadrático, y por qué se prefiere la entropía cruzada para clasificación; la retropropagación como la regla de la cadena organizada en torno a \(\delta^{(l)}\), con las cuatro ecuaciones matriciales; el seguimiento a mano de un paso en la red 2-2-1; la comprobación del gradiente.
- Las decisiones prácticas —tasa de aprendizaje, épocas y detención temprana, inicialización, estandarización, decaimiento de pesos y dropout— y los experimentos que muestran el avance demasiado lento, la divergencia, el sobreajuste y los gradientes que se desvanecen.
- El protocolo del ejemplo resuelto y su resultado: con datos tabulares de unos pocos cientos de casos, una red iguala, pero no supera, a la regresión logística y a los bosques aleatorios; su ventaja está en las entradas en bruto —imágenes, señales, secuencias—, que retoma la Unidad 8.
- Qué agregan la profundidad, la convolución, la recurrencia y la atención al perceptrón multicapa, y por qué importan las GPU.

## 13. Autoevaluación

<details><summary>1. Un colega dice: "nuestra red tiene tres capas: entrada, oculta y salida". ¿Cuántas capas de unidades que calculan tiene, y por qué importa el conteo al leer un artículo?</summary>Dos: la capa de entrada contiene las características y no calcula nada. Los artículos y el software difieren en si la cuentan o no, así que una arquitectura debería darse como la lista de tamaños de capa (p. ej., 9-16-1), que no es ambigua; el número de parámetros, que acota los datos necesarios, se deduce de esa lista.</details>

<details><summary>2. Cuenta los parámetros de una red 64-32-5 e indica qué fracción de los 750 glifos simulados representa ese número.</summary>\(64 \times 32 + 32 + 32 \times 5 + 5 = 2048 + 32 + 160 + 5 = 2245\) parámetros, es decir, unos tres parámetros por imagen de entrenamiento. Una red así puede memorizar el conjunto de entrenamiento; la validación cruzada y el decaimiento de pesos son lo que la mantiene honesta.</details>

<details><summary>3. A mano: una unidad tiene pesos \(w = (2, -1)\), sesgo \(b = -0.5\) y activación sigmoide. Calcula su salida para \(\mathbf{x} = (1, 1)\) y para \(\mathbf{x} = (0, 1)\). ¿En qué lado de la frontera queda cada uno?</summary>\(z = 2 - 1 - 0.5 = 0.5 \Rightarrow a = \sigma(0.5) \approx 0.622\) para \((1,1)\); \(z = 0 - 1 - 0.5 = -1.5 \Rightarrow a = \sigma(-1.5) \approx 0.182\) para \((0,1)\). El primero está en el lado positivo de la recta \(2x_1 - x_2 = 0.5\) (salida mayor que 0.5) y el segundo en el lado negativo.</details>

<details><summary>4. Da pesos y sesgo para una unidad con activación escalón que calcule NOR\((x_1, x_2)\), y explica en una oración por qué ninguna elección calcula XOR.</summary>\(w = (-1, -1)\), \(b = 0.5\): \(z = 0.5\) para \((0,0)\) y negativo en los demás casos. XOR requiere que las dos esquinas diagonales del cuadrado unitario queden a un lado de una recta y las otras dos al otro lado, lo que ninguna recta logra (sumar las desigualdades para \((0,1)\) y \((1,0)\) y restar la de \((0,0)\) contradice la de \((1,1)\)).</details>

<details><summary>5. En el seguimiento a mano del perceptrón de la Sección 7.3, la segunda época comienza con \(\mathbf{w} = (1,1)\), \(b = 0\). Calcula los pesos después de los dos primeros pasos de la época 2 (entradas \((0,0)\) y luego \((0,1)\), \(\eta = 1\)).</summary>Paso 1: \(z = 0 \Rightarrow \hat{y} = +1 \neq -1\); se actualiza \(b \leftarrow -1\), \(\mathbf{w}\) sin cambios \((1,1)\). Paso 2: \(z = 1 - 1 = 0 \Rightarrow \hat{y} = +1 \neq -1\); se actualiza \(\mathbf{w} \leftarrow (1,1) - (0,1) = (1,0)\), \(b \leftarrow -2\). Esto coincide con las filas impresas por el código.</details>

<details><summary>6. El teorema de convergencia del perceptrón acota el número de errores por \((R/\gamma)^2\). ¿Qué le ocurre a la cota si cada característica se multiplica por 10, y qué te dice eso sobre los datos, no sobre el algoritmo?</summary>Nada: \(R\) y \(\gamma\) se multiplican ambos por 10 y la razón no cambia. La cota depende de la <em>geometría</em> del problema —cuán grande es el margen en relación con la dispersión de los datos—, no de sus unidades; un problema cuyas clases casi se tocan (\(\gamma/R\) pequeño) necesita muchas actualizaciones, cualquiera que sea la tasa de aprendizaje.</details>

<details><summary>7. ¿Por qué la función escalón no sirve como activación oculta en una red entrenada por retropropagación, y por qué la sigmoide cedió luego su lugar a la ReLU?</summary>Su derivada es cero en todas partes, de modo que \(\delta^{(l)} = (\dots) \odot \varphi'(z)\) es cero y ningún peso recibe gradiente. La sigmoide es diferenciable, pero su derivada es como máximo 0.25 y se desvanece cuando la unidad se satura, de modo que los gradientes se reducen geométricamente con la profundidad; la derivada de la ReLU es exactamente 1 para las unidades activas, así que los gradientes las atraviesan sin disminuir (Figura 14).</details>

<details><summary>8. En el seguimiento a mano de la retropropagación, los gradientes de la capa oculta fueron unas diez veces menores que los de la capa de salida. Identifica los dos factores responsables e indica cuál elimina la ReLU.</summary>\(\delta^{(1)}_j = \delta^{(2)} W^{(2)}_j\, \sigma'(z^{(1)}_j)\): el error se multiplica por el peso que conecta las unidades (magnitud 0.6 o 0.4) y por la derivada de la sigmoide (como máximo 0.25). La ReLU reemplaza el segundo factor por 1 en las unidades activas; el primer factor se controla con la inicialización.</details>

<details><summary>9. Lee el código: en <code>mlp_backward</code>, ¿por qué <code>delta</code> se inicializa como <code>(A[[length(A)]] - y) / n</code> sin ninguna derivada de la activación, y qué habría que cambiar si la pérdida fuera el error cuadrático?</summary>Para una salida sigmoide con la entropía cruzada binaria, \(\partial L/\partial z^{(L)} = (\hat{y} - y)/n\) exactamente: el factor \(\sigma'(z)\) se cancela con la derivada del logaritmo. Con el error cuadrático, la línea pasaría a ser <code>(A[[length(A)]] - y) * A[[length(A)]] * (1 - A[[length(A)]]) / n</code>, como en el bloque de código de XOR, y el entrenamiento se haría más lento cada vez que la salida se saturara.</details>

<details><summary>10. La Figura 13 muestra que la pérdida de validación sube después de cierto número de épocas mientras la de entrenamiento sigue bajando. Nombra el fenómeno y un remedio, y explica por qué <code>nnet</code> no implementa la detención temprana.</summary>Sobreajuste. Remedios: detención temprana (conservar los pesos de la mejor época de validación), decaimiento de pesos (<code>decay</code>), una red más pequeña o más datos. <code>nnet</code> usa un optimizador cuasi-Newton de lote (BFGS) que corre hasta <code>maxit</code> o hasta su tolerancia en lugar de un SGD época por época, de modo que no hay un punto natural para vigilar un conjunto de validación y detenerse; el bucle programado desde cero, que avanza época por época, sí puede hacerlo.</details>

<details><summary>11. En el ejemplo resuelto, la red, la SVM, el bosque aleatorio y la regresión logística alcanzaron todos un AUC en validación cruzada de alrededor de 0.99. Un revisor te pide justificar el uso de la red. ¿Qué respondes?</summary>Que con estos datos la red no se justifica: no supera al modelo de referencia más simple por más que la variabilidad entre pliegues, y es menos interpretable y más costosa de ajustar. La justificación de las redes está en problemas con entradas en bruto y de alta dimensión (imágenes, señales) donde las características deben aprenderse; las características diseñadas de este conjunto de datos ya contienen la información.</details>

<details><summary>12. La Sección 9.4 estandarizó dentro de la validación cruzada. ¿Qué error introduciría ajustar el escalamiento con el conjunto de datos completo antes de la validación cruzada, y haría que la exactitud reportada fuera mayor o menor que la real?</summary>Fuga de datos: las medias y desviaciones estándar de los pliegues de prueba influirían en la estandarización de los pliegues de entrenamiento, lo que daría al modelo información sobre los datos de prueba. La exactitud reportada tendría un sesgo optimista: sería mayor que la exactitud con casos genuinamente nuevos (Suplemento S1).</details>

## Glosario

<dl class="gloss">
<dt>Función de activación (activation function)</dt><dd>La función no lineal fija \(\varphi\) que se aplica a la suma ponderada de una unidad; escalón, sigmoide, tanh y ReLU para unidades ocultas o de salida, softmax para una capa de salida multiclase.</dd>
<dt>Adaline / regla delta (Adaline / delta rule)</dt><dd>La unidad lineal de Widrow y Hoff entrenada por descenso de gradiente sobre el error cuadrático; su actualización \(\mathbf{w} \leftarrow \mathbf{w} + \eta (y - z)\mathbf{x}\) es el caso de una sola capa de la retropropagación.</dd>
<dt>Arquitectura (architecture)</dt><dd>La especificación de una red: número y tamaño de las capas, funciones de activación, patrón de conexiones (totalmente conectada, convolucional, recurrente).</dd>
<dt>Retropropagación (backpropagation)</dt><dd>El algoritmo que calcula el gradiente de la pérdida respecto de cada peso aplicando la regla de la cadena capa por capa desde la salida hacia atrás, reutilizando \(\delta^{(l)} = \partial L / \partial \mathbf{z}^{(l)}\).</dd>
<dt>Lote, minilote, época (batch, mini-batch, epoch)</dt><dd>El conjunto de casos usado para un paso de gradiente (todos los casos: lote completo; un subconjunto: minilote; un caso: en línea); una pasada por todo el conjunto de entrenamiento es una época.</dd>
<dt>Sesgo (bias)</dt><dd>El término constante \(b\) que se suma a la suma ponderada de una unidad; equivale a un peso sobre una entrada constante igual a 1. Desplaza la frontera de decisión alejándola del origen.</dd>
<dt>Capa convolucional (convolutional layer)</dt><dd>Una capa cuyas unidades ven cada una una pequeña ventana de la entrada y comparten los mismos pesos, de modo que una característica se detecta dondequiera que aparezca; el bloque básico de las redes para imágenes (Unidad 8).</dd>
<dt>Entropía cruzada (cross-entropy)</dt><dd>La pérdida de log-verosimilitud negativa para la clasificación probabilística: binaria para una salida sigmoide, categórica para una salida softmax. Con estas salidas, su gradiente respecto de los logits es \(\hat{y} - y\).</dd>
<dt>Aprendizaje profundo (deep learning)</dt><dd>Redes neuronales con muchas capas ocultas, entrenadas de extremo a extremo a partir de entradas en bruto; la jerarquía aprendida de representaciones es su rasgo distintivo.</dd>
<dt>Dropout (dropout)</dt><dd>Un método de regularización que lleva a cero cada unidad oculta al azar durante el entrenamiento, lo que obliga a formar representaciones redundantes.</dd>
<dt>Detención temprana (early stopping)</dt><dd>Terminar el entrenamiento cuando la pérdida en un conjunto de validación deja de mejorar y conservar los pesos de la mejor época; el regularizador más común para redes.</dd>
<dt>Prealimentada / recurrente (feed-forward / recurrent)</dt><dd>Una red sin ciclos, cuya salida depende solo de la entrada actual / una red con conexiones de retroalimentación, cuya salida depende de la secuencia de entradas pasadas.</dd>
<dt>Inicialización de Glorot o Xavier (Glorot (Xavier) initialisation)</dt><dd>Pesos iniciales aleatorios con varianza \(2/(n_{\text{in}} + n_{\text{out}})\), elegida para que la varianza de la señal y del gradiente se mantenga aproximadamente constante a través de las capas.</dd>
<dt>Comprobación del gradiente (gradient checking)</dt><dd>Verificar una implementación de la retropropagación comparando sus derivadas con diferencias finitas centradas de la pérdida.</dd>
<dt>Aprendizaje hebbiano (Hebbian learning)</dt><dd>La regla de Hebb de 1949: una sinapsis se fortalece cuando las neuronas presináptica y postsináptica están activas a la vez; la primera propuesta de que aprender es ajustar la fuerza de las conexiones.</dd>
<dt>Capa oculta (hidden layer)</dt><dd>Una capa cuyas activaciones no son ni entradas ni salidas; sus unidades calculan las características aprendidas con las que opera la capa de salida.</dd>
<dt>Hiperparámetro (hyperparameter)</dt><dd>Una magnitud fijada antes del entrenamiento (arquitectura, tasa de aprendizaje, intensidad de la penalización, épocas) y elegida por validación cruzada, a diferencia de un parámetro aprendido por descenso de gradiente.</dd>
<dt>Tasa de aprendizaje (learning rate)</dt><dd>El tamaño de paso \(\eta\) del descenso de gradiente; si es demasiado pequeña, avanza muy lento; si es demasiado grande, diverge.</dd>
<dt>Linealmente separable (linearly separable)</dt><dd>Una muestra etiquetada para la cual algún hiperplano deja todos los casos de una clase a un lado y todos los de la otra al otro lado; la condición bajo la cual el perceptrón converge.</dd>
<dt>Logit (logit)</dt><dd>La preactivación de una unidad de salida, \(z = \mathbf{w}^\top \mathbf{a} + b\), antes de la sigmoide o la softmax; igual al logaritmo de las odds (log-odds) para una salida sigmoide.</dd>
<dt>Neurona de McCulloch–Pitts (McCulloch–Pitts neuron)</dt><dd>La unidad de 1943: entradas binarias, un umbral y una salida binaria; el primer modelo formal de una neurona como dispositivo lógico.</dd>
<dt>Perceptrón multicapa (multilayer perceptron, MLP)</dt><dd>Una red prealimentada totalmente conectada, con una o más capas ocultas y activaciones diferenciables, entrenada por retropropagación.</dd>
<dt>Perceptrón (perceptron)</dt><dd>La unidad de umbral de Rosenblatt y su regla de aprendizaje: actualizar \(\mathbf{w} \leftarrow \mathbf{w} + \eta y \mathbf{x}\) solo en los ejemplos mal clasificados.</dd>
<dt>Teorema de convergencia del perceptrón (perceptron convergence theorem)</dt><dd>El resultado de Novikoff: con datos linealmente separables con margen \(\gamma\) y radio \(R\), el perceptrón comete como máximo \((R/\gamma)^2\) errores.</dd>
<dt>ReLU (ReLU)</dt><dd>Unidad lineal rectificada (rectified linear unit), \(\varphi(z) = \max(0, z)\); la activación oculta por defecto de las redes profundas, porque su derivada es 1 para las unidades activas.</dd>
<dt>Saturación (saturation)</dt><dd>El régimen en el que la preactivación de una unidad sigmoide o tanh es grande en magnitud, su salida está en un extremo y su derivada es cercana a cero, de modo que poco gradiente pasa a través de ella.</dd>
<dt>Sigmoide (sigmoid)</dt><dd>La función logística \(\sigma(z) = 1/(1 + e^{-z})\) (<code>plogis</code> de R); la activación de salida para la clasificación binaria, que entrega \(P(y = 1 \mid \mathbf{x})\).</dd>
<dt>Softmax (softmax)</dt><dd>La función que transforma \(K\) logits en \(K\) números positivos que suman 1; la activación de salida para la clasificación multiclase.</dd>
<dt>Teorema de aproximación universal (universal approximation theorem)</dt><dd>El resultado de Cybenko y de Hornik según el cual una red con una capa oculta de suficientes unidades no lineales puede aproximar cualquier función continua sobre un dominio acotado; un resultado de existencia que no dice nada sobre la aprendibilidad.</dd>
<dt>Gradiente que se desvanece (vanishing gradient)</dt><dd>La reducción geométrica de los gradientes a medida que se propagan hacia atrás a través de muchas capas que se saturan (o de muchos pasos de tiempo), de modo que las primeras capas apenas aprenden; se mitiga con ReLU, inicialización escalada y atajos arquitectónicos.</dd>
<dt>Peso (weight)</dt><dd>El número real ajustable de una conexión, que multiplica la señal que pasa por ella; los parámetros de una red son sus pesos y sesgos.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">McCulloch WS, Pitts W. A logical calculus of the ideas immanent in nervous activity. <em>Bull Math Biophys</em>. 1943;5:115–133. <a href="https://doi.org/10.1007/BF02478259">doi:10.1007/BF02478259</a></li>
<li id="r2">Rosenblatt F. The perceptron: a probabilistic model for information storage and organization in the brain. <em>Psychol Rev</em>. 1958;65(6):386–408. <a href="https://doi.org/10.1037/h0042519">doi:10.1037/h0042519</a></li>
<li id="r3">Minsky M, Papert S. <em>Perceptrons: An Introduction to Computational Geometry</em>. MIT Press; 1969. Reeditado en 2017. <a href="https://doi.org/10.7551/mitpress/11301.001.0001">doi:10.7551/mitpress/11301.001.0001</a></li>
<li id="r4">Rumelhart DE, Hinton GE, Williams RJ. Learning representations by back-propagating errors. <em>Nature</em>. 1986;323:533–536. <a href="https://doi.org/10.1038/323533a0">doi:10.1038/323533a0</a></li>
<li id="r5">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84–90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r6">LeCun Y, Bengio Y, Hinton G. Deep learning. <em>Nature</em>. 2015;521:436–444. <a href="https://doi.org/10.1038/nature14539">doi:10.1038/nature14539</a></li>
<li id="r7">Azevedo FAC, Carvalho LRB, Grinberg LT, et al. Equal numbers of neuronal and nonneuronal cells make the human brain an isometrically scaled-up primate brain. <em>J Comp Neurol</em>. 2009;513(5):532–541. <a href="https://doi.org/10.1002/cne.21974">doi:10.1002/cne.21974</a></li>
<li id="r8">Herculano-Houzel S. The human brain in numbers: a linearly scaled-up primate brain. <em>Front Hum Neurosci</em>. 2009;3:31. <a href="https://doi.org/10.3389/neuro.09.031.2009">doi:10.3389/neuro.09.031.2009</a></li>
<li id="r9">Pakkenberg B, Pelvig D, Marner L, et al. Aging and the human neocortex. <em>Exp Gerontol</em>. 2003;38(1–2):95–99. <a href="https://doi.org/10.1016/S0531-5565(02)00151-1">doi:10.1016/S0531-5565(02)00151-1</a></li>
<li id="r10">Hodgkin AL, Huxley AF. A quantitative description of membrane current and its application to conduction and excitation in nerve. <em>J Physiol</em>. 1952;117(4):500–544. <a href="https://doi.org/10.1113/jphysiol.1952.sp004764">doi:10.1113/jphysiol.1952.sp004764</a></li>
<li id="r11">Brunel N, van Rossum MCW. Lapicque's 1907 paper: from frogs to integrate-and-fire. <em>Biol Cybern</em>. 2007;97:337–339. <a href="https://doi.org/10.1007/s00422-007-0190-0">doi:10.1007/s00422-007-0190-0</a></li>
<li id="r12">Hebb DO. <em>The Organization of Behavior: A Neuropsychological Theory</em>. New York: Wiley; 1949. Reeditado por Psychology Press; 2002. <a href="https://doi.org/10.4324/9781410612403">doi:10.4324/9781410612403</a></li>
<li id="r13">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r14">Cybenko G. Approximation by superpositions of a sigmoidal function. <em>Math Control Signals Syst</em>. 1989;2:303–314. <a href="https://doi.org/10.1007/BF02551274">doi:10.1007/BF02551274</a></li>
<li id="r15">Hornik K, Stinchcombe M, White H. Multilayer feedforward networks are universal approximators. <em>Neural Netw</em>. 1989;2(5):359–366. <a href="https://doi.org/10.1016/0893-6080(89)90020-8">doi:10.1016/0893-6080(89)90020-8</a></li>
<li id="r16">Baker JA, Kornguth PJ, Lo JY, Williford ME, Floyd CE. Breast cancer: prediction with artificial neural network based on BI-RADS standardized lexicon. <em>Radiology</em>. 1995;196(3):817–822. <a href="https://doi.org/10.1148/radiology.196.3.7644649">doi:10.1148/radiology.196.3.7644649</a></li>
<li id="r17">Lisboa PJ, Taktak AFG. The use of artificial neural networks in decision support in cancer: a systematic review. <em>Neural Netw</em>. 2006;19(4):408–415. <a href="https://doi.org/10.1016/j.neunet.2005.10.007">doi:10.1016/j.neunet.2005.10.007</a></li>
<li id="r18">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402–2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r19">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115–118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r20">De Fauw J, Ledsam JR, Romera-Paredes B, et al. Clinically applicable deep learning for diagnosis and referral in retinal disease. <em>Nat Med</em>. 2018;24:1342–1350. <a href="https://doi.org/10.1038/s41591-018-0107-6">doi:10.1038/s41591-018-0107-6</a></li>
<li id="r21">Hannun AY, Rajpurkar P, Haghpanahi M, et al. Cardiologist-level arrhythmia detection and classification in ambulatory electrocardiograms using a deep neural network. <em>Nat Med</em>. 2019;25:65–69. <a href="https://doi.org/10.1038/s41591-018-0268-3">doi:10.1038/s41591-018-0268-3</a></li>
<li id="r22">Rajkomar A, Dean J, Kohane I. Machine learning in medicine. <em>N Engl J Med</em>. 2019;380(14):1347–1358. <a href="https://doi.org/10.1056/NEJMra1814259">doi:10.1056/NEJMra1814259</a></li>
<li id="r23">Novikoff AB. On convergence proofs on perceptrons. In: <em>Proceedings of the Symposium on the Mathematical Theory of Automata</em>, vol. XII. Brooklyn, NY: Polytechnic Institute of Brooklyn; 1962:615–622. (Actas de congreso sin DOI; el teorema y su demostración se reproducen en Minsky y Papert [3] y en <em>Neural Networks for Pattern Recognition</em> de Bishop, incluido en Libros de texto).</li>
<li id="r24">Widrow B, Hoff ME. Adaptive switching circuits. <em>1960 IRE WESCON Convention Record</em>, part 4. New York: IRE; 1960:96–104. Stanford Electronics Laboratories Technical Report 1553-1. <a href="https://doi.org/10.21236/AD0241531">doi:10.21236/AD0241531</a></li>
<li id="r25">Glorot X, Bengio Y. Understanding the difficulty of training deep feedforward neural networks. <em>Proceedings of the 13th International Conference on Artificial Intelligence and Statistics (AISTATS)</em>, PMLR 9:249–256; 2010. <a href="https://proceedings.mlr.press/v9/glorot10a.html">proceedings.mlr.press/v9/glorot10a.html</a> (las actas de PMLR no tienen DOI)</li>
<li id="r26">Nair V, Hinton GE. Rectified linear units improve restricted Boltzmann machines. <em>Proceedings of the 27th International Conference on Machine Learning (ICML)</em>; 2010:807–814. <a href="https://icml.cc/Conferences/2010/papers/432.pdf">icml.cc/Conferences/2010/papers/432.pdf</a> (las actas de ICML 2010 no tienen DOI)</li>
<li id="r27">Srivastava N, Hinton G, Krizhevsky A, Sutskever I, Salakhutdinov R. Dropout: a simple way to prevent neural networks from overfitting. <em>J Mach Learn Res</em>. 2014;15(56):1929–1958. <a href="https://jmlr.org/papers/v15/srivastava14a.html">jmlr.org/papers/v15/srivastava14a.html</a> (los artículos de JMLR no tienen DOI)</li>
<li id="r28">Kingma DP, Ba J. Adam: a method for stochastic optimization. <em>3rd International Conference on Learning Representations (ICLR)</em>; 2015. <a href="https://doi.org/10.48550/arXiv.1412.6980">doi:10.48550/arXiv.1412.6980</a></li>
<li id="r29">Hochreiter S. The vanishing gradient problem during learning recurrent neural nets and problem solutions. <em>Int J Uncertain Fuzziness Knowl-Based Syst</em>. 1998;6(2):107–116. <a href="https://doi.org/10.1142/S0218488598000094">doi:10.1142/S0218488598000094</a></li>
<li id="r30">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. (Fuente del conjunto de datos de diabetes de sklearn usado en la edición en Python). <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r31">Alpaydin E, Kaynak C. Cascading classifiers. <em>Kybernetika</em>. 1998;34(4):369–374. Conjunto de datos UCI Optical Recognition of Handwritten Digits (usado en la edición en Python; simulado en R). <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r32">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. <em>Proceedings of the 22nd International Conference on Machine Learning (ICML)</em>; 2005:625–632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r33">LeCun Y, Bottou L, Bengio Y, Haffner P. Gradient-based learning applied to document recognition. <em>Proc IEEE</em>. 1998;86(11):2278–2324. <a href="https://doi.org/10.1109/5.726791">doi:10.1109/5.726791</a></li>
<li id="r34">Hochreiter S, Schmidhuber J. Long short-term memory. <em>Neural Comput</em>. 1997;9(8):1735–1780. <a href="https://doi.org/10.1162/neco.1997.9.8.1735">doi:10.1162/neco.1997.9.8.1735</a></li>
<li id="r35">Vaswani A, Shazeer N, Parmar N, et al. Attention is all you need. <em>Advances in Neural Information Processing Systems 30 (NeurIPS 2017)</em>. <a href="https://doi.org/10.48550/arXiv.1706.03762">doi:10.48550/arXiv.1706.03762</a></li>
<li id="r36">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. (La referencia para <code>nnet</code>, <code>MASS</code> y las funciones de modelado de R usadas aquí). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r37">Fritsch S, Guenther F, Wright MN. neuralnet: Training of Neural Networks. Paquete de R. <a href="https://doi.org/10.32614/CRAN.package.neuralnet">doi:10.32614/CRAN.package.neuralnet</a></li>
<li id="r38">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R (fuente de los datos de cáncer de mama de Wisconsin y de Boston housing usados aquí). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. (El compañero en R de todo el curso; PDF gratuito en statlearning.com). <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>Bishop CM. <em>Neural Networks for Pattern Recognition</em>. Oxford University Press; 1995. — El tratamiento clásico del perceptrón multicapa, la retropropagación y su interpretación estadística.</li>
<li>Goodfellow I, Bengio Y, Courville A. <em>Deep Learning</em>. MIT Press; 2016. — Los Capítulos 6–8 amplían las Secciones 8–9; gratuito en línea en deeplearningbook.org.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulo 11 (redes neuronales) para la mirada del estadístico. PDF gratuito.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS)<sup class="c"><a href="#r36">[36]</a></sup> — la referencia para <code>nnet</code> y las funciones de modelado de R; y James et al., <em>Introduction to Statistical Learning</em><sup class="c"><a href="#r39">[39]</a></sup>, como compañero en R del curso.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Los manuales de los paquetes <code>nnet</code> y <code>neuralnet</code><sup class="c"><a href="#r37">[37]</a></sup> en CRAN: documentación de las funciones de entrenamiento usadas a partir de la Sección 9.</li>
<li>Nielsen M. <em>Neural Networks and Deep Learning</em> (libro en línea) — <a href="http://neuralnetworksanddeeplearning.com">neuralnetworksanddeeplearning.com</a>; una exposición de la retropropagación desde cero, en el espíritu de la Sección 8.</li>
</ul>
